# Feature Engineering

Este notebook mostra a primeira camada de features comuns do projeto Prioriza-Leitos.

A ideia é entender, de forma auditável, como cada atributo ajuda a responder a pergunta central:

> Qual município apresenta maior prioridade relativa para investigação e possível investimento em determinado tipo de leito SUS?

Importante: nenhuma feature deve ser lida isoladamente como recomendação automática. Em especial, `municipio_sem_leito_do_tipo` indica ausência de oferta local, mas não significa, sozinha, que o município deva receber novos leitos.

Unidade da tabela de features:

`municipio + ano + tipo_leito`

Atualização registrada em `2026-06-07`.

## Setup

Primeiro carregamos bibliotecas, localizamos a raiz do projeto e lemos a tabela de features. Se a tabela ainda não existir, o notebook tenta reconstruí-la a partir das facts em `data/processed/`.

In [1]:
from pathlib import Path
import sys

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / "pyproject.toml").exists() and (path / "src").exists():
            return path
    raise RuntimeError("Raiz do projeto não encontrada.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
FEATURES_PATH = PROCESSED_DIR / "features_common_municipio_ano_tipo_leito.parquet"
CLINICAL_FEATURES_PATH = PROCESSED_DIR / "features_leitos_clinicos.parquet"
SURGICAL_FEATURES_PATH = PROCESSED_DIR / "features_leitos_cirurgicos.parquet"
OBSTETRIC_FEATURES_PATH = PROCESSED_DIR / "features_leitos_obstetricos.parquet"
UTI_ADULTO_FEATURES_PATH = PROCESSED_DIR / "features_uti_adulto.parquet"
UTI_PEDIATRICA_FEATURES_PATH = PROCESSED_DIR / "features_uti_pediatrica.parquet"
UTI_NEONATAL_FEATURES_PATH = PROCESSED_DIR / "features_uti_neonatal.parquet"
UTI_QUEIMADO_REGIONAL_FEATURES_PATH = PROCESSED_DIR / "features_uti_queimado_regional.parquet"
UTI_CORONARIANA_FEATURES_PATH = PROCESSED_DIR / "features_uti_coronariana.parquet"
DIM_PATH = PROCESSED_DIR / "dim_municipio_rs.parquet"

PROJECT_ROOT

PosixPath('/home/berr/GitHub/Prioriza-Leitos')

In [2]:
if not FEATURES_PATH.exists():
    from src.features.build_common_features import build_common_features

    build_common_features(output_path=FEATURES_PATH)

if not CLINICAL_FEATURES_PATH.exists():
    from src.features.build_features_general_beds import build_features_leitos_clinicos

    build_features_leitos_clinicos(output_path=CLINICAL_FEATURES_PATH)

if not SURGICAL_FEATURES_PATH.exists():
    from src.features.build_features_general_beds import build_features_leitos_cirurgicos

    build_features_leitos_cirurgicos(output_path=SURGICAL_FEATURES_PATH)

if not OBSTETRIC_FEATURES_PATH.exists():
    from src.features.build_features_general_beds import build_features_leitos_obstetricos

    build_features_leitos_obstetricos(output_path=OBSTETRIC_FEATURES_PATH)

if not UTI_ADULTO_FEATURES_PATH.exists():
    from src.features.build_features_uti_adulto import build_features_uti_adulto

    build_features_uti_adulto(output_path=UTI_ADULTO_FEATURES_PATH)

if not UTI_PEDIATRICA_FEATURES_PATH.exists():
    from src.features.build_features_uti_pediatrica import build_features_uti_pediatrica

    build_features_uti_pediatrica(output_path=UTI_PEDIATRICA_FEATURES_PATH)

if not UTI_NEONATAL_FEATURES_PATH.exists():
    from src.features.build_features_uti_neonatal import build_features_uti_neonatal

    build_features_uti_neonatal(output_path=UTI_NEONATAL_FEATURES_PATH)

if not UTI_QUEIMADO_REGIONAL_FEATURES_PATH.exists():
    from src.features.build_features_uti_queimado import build_features_uti_queimado_regional

    build_features_uti_queimado_regional(output_path=UTI_QUEIMADO_REGIONAL_FEATURES_PATH)

if not UTI_CORONARIANA_FEATURES_PATH.exists():
    from src.features.build_features_uti_coronariana import build_features_uti_coronariana

    build_features_uti_coronariana(output_path=UTI_CORONARIANA_FEATURES_PATH)

features = pd.read_parquet(FEATURES_PATH)
features_clinicos = pd.read_parquet(CLINICAL_FEATURES_PATH)
features_cirurgicos = pd.read_parquet(SURGICAL_FEATURES_PATH)
features_obstetricos = pd.read_parquet(OBSTETRIC_FEATURES_PATH)
features_uti_adulto = pd.read_parquet(UTI_ADULTO_FEATURES_PATH)
features_uti_pediatrica = pd.read_parquet(UTI_PEDIATRICA_FEATURES_PATH)
features_uti_neonatal = pd.read_parquet(UTI_NEONATAL_FEATURES_PATH)
features_uti_queimado_regional = pd.read_parquet(UTI_QUEIMADO_REGIONAL_FEATURES_PATH)
features_uti_coronariana = pd.read_parquet(UTI_CORONARIANA_FEATURES_PATH)
dim = pd.read_parquet(DIM_PATH).rename(columns={"municipio_id_ibge": "municipio_id"})

features.shape

(17395, 14)

## O que existe nesta tabela?

A tabela reúne features comuns a todos os modelos por tipo de leito do MVP.

As chaves são:

- `municipio_id`
- `ano`
- `tipo_leito`

As demais colunas são sinais de oferta, demanda, pressão assistencial, evolução temporal, distância e fluxo regional.

In [3]:
features.head(10)

,municipio_id,ano,tipo_leito,populacao_total,leitos_sus_por_1000_hab,internacoes_por_1000_hab,dias_permanencia_por_1000_hab,taxa_mortalidade_hospitalar,media_permanencia,crescimento_internacoes_3anos,crescimento_populacao_3anos,distancia_ate_polo_hospitalar,saldo_pacientes_regional,municipio_sem_leito_do_tipo
0,4300034,2019,leitos_cirurgicos,4901,1.42828,12.650479,44.684758,0.016129,3.532258,NaN,<NA>,0.0,-5,False
1,4300034,2019,leitos_clinicos,4901,3.0606,59.987758,251.78535,0.027211,4.197279,NaN,<NA>,0.0,126,False
2,4300034,2019,leitos_obstetricos,4901,1.0202,7.54948,15.507039,0.000000,2.054054,NaN,<NA>,0.0,-13,False
3,4300034,2019,leitos_pediatricos,4901,1.17323,18.159559,69.169557,0.011236,3.808989,NaN,<NA>,0.0,22,False
4,4300034,2019,uti_adulto,4901,0.0,3.67272,35.706999,0.111111,9.722222,NaN,<NA>,0.0,-18,True
5,4300034,2020,leitos_cirurgicos,4942,1.416431,14.771348,50.991501,0.000000,3.452055,NaN,<NA>,0.0,-11,False
6,4300034,2020,leitos_clinicos,4942,3.035208,39.660057,170.578713,0.071429,4.301020,NaN,<NA>,0.0,96,False
7,4300034,2020,leitos_obstetricos,4942,1.011736,7.891542,18.008903,0.000000,2.282051,NaN,<NA>,0.0,-5,False
8,4300034,2020,leitos_pediatricos,4942,1.214083,8.498584,53.014974,0.023810,6.238095,NaN,<NA>,0.0,-10,False
9,4300034,2020,uti_adulto,4942,0.0,2.832861,28.328612,0.357143,10.000000,NaN,<NA>,0.0,-14,True


In [20]:
feature_catalog = pd.DataFrame(
    [
        {
            "feature": "populacao_total",
            "grupo": "denominador",
            "interpretação": "Tamanho populacional do município no ano.",
            "cuidado": "É denominador geral; alguns tipos de leito pedem denominadores específicos em etapa futura.",
        },
        {
            "feature": "leitos_sus_por_1000_hab",
            "grupo": "oferta",
            "interpretação": "Oferta média anual de leitos SUS do tipo por 1.000 habitantes.",
            "cuidado": "Usa média mensal de QT_SUS; não mede qualidade nem disponibilidade real diária.",
        },
        {
            "feature": "internacoes_por_1000_hab",
            "grupo": "demanda",
            "interpretação": "Internações de residentes associadas ao tipo de leito por 1.000 habitantes.",
            "cuidado": "O tipo de leito no SIH é proxy, não campo direto da AIH.",
        },
        {
            "feature": "dias_permanencia_por_1000_hab",
            "grupo": "pressão assistencial",
            "interpretação": "Carga de dias de internação de residentes por 1.000 habitantes.",
            "cuidado": "Pode subir por maior volume, maior gravidade ou permanências mais longas.",
        },
        {
            "feature": "taxa_mortalidade_hospitalar",
            "grupo": "risco/desfecho",
            "interpretação": "Proporção de internações que terminaram em óbito hospitalar.",
            "cuidado": "Não é causal; depende de gravidade, perfil dos casos e acesso.",
        },
        {
            "feature": "media_permanencia",
            "grupo": "pressão assistencial",
            "interpretação": "Média de dias de permanência por internação.",
            "cuidado": "Valores altos podem refletir complexidade, gargalo de alta ou perfil clínico.",
        },
        {
            "feature": "crescimento_internacoes_3anos",
            "grupo": "tendência",
            "interpretação": "Variação percentual das internações contra o mesmo município e tipo de leito três anos antes.",
            "cuidado": "Fica ausente quando não existe ano anterior comparável ou base zero.",
        },
        {
            "feature": "crescimento_populacao_3anos",
            "grupo": "tendência",
            "interpretação": "Variação percentual da população contra três anos antes.",
            "cuidado": "É repetida por tipo de leito porque a unidade da tabela inclui tipo_leito.",
        },
        {
            "feature": "distancia_ate_polo_hospitalar",
            "grupo": "acesso espacial",
            "interpretação": "Distância em km até o município mais próximo com proxy de referência hospitalar.",
            "cuidado": "Usa distância geográfica em linha reta; não substitui tempo de deslocamento ou rede pactuada.",
        },
        {
            "feature": "saldo_pacientes_regional",
            "grupo": "fluxo regional",
            "interpretação": "Pacientes recebidos de outros municípios menos residentes internados fora.",
            "cuidado": "Saldo positivo sugere polo receptor; saldo negativo sugere dependência externa.",
        },
        {
            "feature": "municipio_sem_leito_do_tipo",
            "grupo": "oferta",
            "interpretação": "Indica se não houve leito SUS do tipo no município ao longo do ano.",
            "cuidado": "Não deve virar prioridade automática; é apenas um sinal combinado com demanda, fluxo e distância.",
        },
    ]
)

feature_catalog

,feature,grupo,interpretação,cuidado
0,populacao_total,denominador,Tamanho populacional do município no ano.,É denominador geral; alguns tipos de leito ped...
1,leitos_sus_por_1000_hab,oferta,Oferta média anual de leitos SUS do tipo por 1...,Usa média mensal de QT_SUS; não mede qualidade...
2,internacoes_por_1000_hab,demanda,Internações de residentes associadas ao tipo d...,"O tipo de leito no SIH é proxy, não campo dire..."
3,dias_permanencia_por_1000_hab,pressão assistencial,Carga de dias de internação de residentes por ...,"Pode subir por maior volume, maior gravidade o..."
4,taxa_mortalidade_hospitalar,risco/desfecho,Proporção de internações que terminaram em óbi...,"Não é causal; depende de gravidade, perfil dos..."
5,media_permanencia,pressão assistencial,Média de dias de permanência por internação.,"Valores altos podem refletir complexidade, gar..."
6,crescimento_internacoes_3anos,tendência,Variação percentual das internações contra o m...,Fica ausente quando não existe ano anterior co...
7,crescimento_populacao_3anos,tendência,Variação percentual da população contra três a...,É repetida por tipo de leito porque a unidade ...
8,distancia_ate_polo_hospitalar,acesso espacial,Distância em km até o município mais próximo c...,Usa distância geográfica em linha reta; não su...
9,saldo_pacientes_regional,fluxo regional,Pacientes recebidos de outros municípios menos...,Saldo positivo sugere polo receptor; saldo neg...


## Validações de cobertura

Antes de interpretar qualquer ranking, vale checar se a tabela cobre a granularidade esperada: municípios do RS, anos do MVP e tipos de leito do MVP.

In [5]:
coverage = {
    "linhas": len(features),
    "municipios": features["municipio_id"].nunique(),
    "anos": sorted(features["ano"].unique().tolist()),
    "tipos_leito": sorted(features["tipo_leito"].unique().tolist()),
    "chave_duplicada": features.duplicated(["municipio_id", "ano", "tipo_leito"]).any(),
}

coverage

{'linhas': 17395,
 'municipios': 497,
 'anos': [2019, 2020, 2021, 2022, 2023, 2024, 2025],
 'tipos_leito': ['leitos_cirurgicos',
  'leitos_clinicos',
  'leitos_obstetricos',
  'leitos_pediatricos',
  'uti_adulto'],
 'chave_duplicada': False}

In [6]:
missing = (
    features.isna()
    .sum()
    .rename("valores_ausentes")
    .to_frame()
    .assign(percentual=lambda df: df["valores_ausentes"] / len(features))
)

missing

,valores_ausentes,percentual
municipio_id,0,0.000000
ano,0,0.000000
tipo_leito,0,0.000000
populacao_total,0,0.000000
leitos_sus_por_1000_hab,0,0.000000
internacoes_por_1000_hab,0,0.000000
dias_permanencia_por_1000_hab,0,0.000000
taxa_mortalidade_hospitalar,8,0.000460
media_permanencia,8,0.000460
crescimento_internacoes_3anos,7462,0.428974


Ausências esperadas:

- `crescimento_internacoes_3anos` e `crescimento_populacao_3anos`: os três primeiros anos não têm lag de três anos.
- `taxa_mortalidade_hospitalar` e `media_permanencia`: ficam ausentes quando não há internações na combinação `município + ano + tipo_leito`.

## Como o SIH vira tipo de leito?

O SIH não traz diretamente o leito ocupado como `leitos_clinicos`, `leitos_cirurgicos`, `leitos_obstetricos`, `leitos_pediatricos` ou `uti_adulto`.

Por isso, esta primeira camada usa proxies transparentes:

- `procedimentos_clinicos` -> `leitos_clinicos`
- `procedimentos_cirurgicos` -> `leitos_cirurgicos`
- CID principal começando com `O` ou procedimento obstétrico -> `leitos_obstetricos`
- idade menor que 15 anos -> `leitos_pediatricos`
- uso de UTI em paciente adulto -> `uti_adulto`

Essa regra é simples de auditar, mas ainda é uma proxy. Ela deve ser documentada e refinada com SIGTAP/CID quando o projeto avançar.

In [7]:
from src.features.build_common_features import classify_sih_bed_type

exemplo_sih = pd.DataFrame(
    {
        "grupo_procedimento": [
            "procedimentos_clinicos",
            "procedimentos_cirurgicos",
            "procedimentos_clinicos",
            "procedimentos_clinicos",
            "procedimentos_clinicos",
        ],
        "diagnostico_cid": ["J189", "K359", "O800", "A085", "J960"],
        "procedimento_codigo": ["0303010010", "0407030026", "0310010039", "0303010061", "0303010010"],
        "idade_dias": [40 * 365, 40 * 365, 25 * 365, 4 * 365, 70 * 365],
        "uso_uti": [False, False, False, False, True],
    }
)

exemplo_sih.assign(tipo_leito_proxy=classify_sih_bed_type(exemplo_sih))

,grupo_procedimento,diagnostico_cid,procedimento_codigo,idade_dias,uso_uti,tipo_leito_proxy
0,procedimentos_clinicos,J189,0303010010,14600,False,leitos_clinicos
1,procedimentos_cirurgicos,K359,0407030026,14600,False,leitos_cirurgicos
2,procedimentos_clinicos,O800,0310010039,9125,False,leitos_obstetricos
3,procedimentos_clinicos,A085,0303010061,1460,False,leitos_pediatricos
4,procedimentos_clinicos,J960,0303010010,25550,True,uti_adulto


## Resumo por tipo de leito

Aqui a pergunta não é ainda "quem deve receber leito". A pergunta é: como cada tipo de leito se comporta no conjunto de features?

In [8]:
features.groupby("tipo_leito").agg(
    municipios=("municipio_id", "nunique"),
    anos=("ano", "nunique"),
    media_leitos_sus_1000=("leitos_sus_por_1000_hab", "mean"),
    media_internacoes_1000=("internacoes_por_1000_hab", "mean"),
    media_dias_perm_1000=("dias_permanencia_por_1000_hab", "mean"),
    media_mortalidade_hospitalar=("taxa_mortalidade_hospitalar", "mean"),
    municipios_ano_sem_leito=("municipio_sem_leito_do_tipo", "sum"),
).sort_values("media_internacoes_1000", ascending=False)

,municipios,anos,media_leitos_sus_1000,media_internacoes_1000,media_dias_perm_1000,media_mortalidade_hospitalar,municipios_ano_sem_leito
tipo_leito,,,,,,,
leitos_clinicos,497,7,0.909143,34.549443,228.820947,0.087381,1837
leitos_cirurgicos,497,7,0.198201,25.712376,60.140591,0.007618,2091
leitos_obstetricos,497,7,0.132743,8.150686,19.907214,0.000267,2125
leitos_pediatricos,497,7,0.187698,7.662124,39.725293,0.012383,1948
uti_adulto,497,7,0.027127,4.558959,48.825319,0.263934,3061


In [9]:
numeric_features = [
    "populacao_total",
    "leitos_sus_por_1000_hab",
    "internacoes_por_1000_hab",
    "dias_permanencia_por_1000_hab",
    "taxa_mortalidade_hospitalar",
    "media_permanencia",
    "crescimento_internacoes_3anos",
    "crescimento_populacao_3anos",
    "distancia_ate_polo_hospitalar",
    "saldo_pacientes_regional",
]

features[numeric_features].describe().T

,count,mean,std,min,25%,50%,75%,max
populacao_total,17395.0,22562.791032,77692.731527,932.0,2919.0,5499.0,15414.0,1492530.0
leitos_sus_por_1000_hab,17395.0,0.290983,0.78064,0.0,0.0,0.0,0.25176,15.673981
internacoes_por_1000_hab,17395.0,16.126718,15.41811,0.0,5.437108,9.692132,23.35263,159.827214
dias_permanencia_por_1000_hab,17395.0,79.483873,90.457024,0.0,25.159531,45.96835,86.019485,1041.479821
taxa_mortalidade_hospitalar,17387.0,0.074274,0.117937,0.0,0.0,0.011801,0.102041,1.0
media_permanencia,17387.0,5.621662,3.692515,0.0,2.496512,4.744186,7.901389,39.5
crescimento_internacoes_3anos,9933.0,0.283488,0.75131,-1.0,-0.081866,0.135593,0.448276,13.8
crescimento_populacao_3anos,9940.0,-0.006711,0.077712,-0.306943,-0.053776,0.01243,0.028343,0.505822
distancia_ate_polo_hospitalar,17395.0,10.035708,12.022604,0.0,0.0,9.380961,15.807545,79.199624
saldo_pacientes_regional,17395.0,0.0,775.077364,-4300.0,-88.0,-36.0,-12.0,28544.0


## Exemplos com nome do município

Para leitura humana, juntamos a dimensão municipal. Isso ajuda a entender rankings sem decorar código IBGE.

In [10]:
features_named = features.merge(
    dim[["municipio_id", "municipio_nome", "regiao_saude", "macroregiao_saude"]],
    on="municipio_id",
    how="left",
    validate="many_to_one",
)

features_named.head()

,municipio_id,ano,tipo_leito,populacao_total,leitos_sus_por_1000_hab,internacoes_por_1000_hab,dias_permanencia_por_1000_hab,taxa_mortalidade_hospitalar,media_permanencia,crescimento_internacoes_3anos,crescimento_populacao_3anos,distancia_ate_polo_hospitalar,saldo_pacientes_regional,municipio_sem_leito_do_tipo,municipio_nome,regiao_saude,macroregiao_saude
0,4300034,2019,leitos_cirurgicos,4901,1.42828,12.650479,44.684758,0.016129,3.532258,NaN,<NA>,0.0,-5,False,Aceguá,22 - Pampa,Sul
1,4300034,2019,leitos_clinicos,4901,3.0606,59.987758,251.78535,0.027211,4.197279,NaN,<NA>,0.0,126,False,Aceguá,22 - Pampa,Sul
2,4300034,2019,leitos_obstetricos,4901,1.0202,7.54948,15.507039,0.000000,2.054054,NaN,<NA>,0.0,-13,False,Aceguá,22 - Pampa,Sul
3,4300034,2019,leitos_pediatricos,4901,1.17323,18.159559,69.169557,0.011236,3.808989,NaN,<NA>,0.0,22,False,Aceguá,22 - Pampa,Sul
4,4300034,2019,uti_adulto,4901,0.0,3.67272,35.706999,0.111111,9.722222,NaN,<NA>,0.0,-18,True,Aceguá,22 - Pampa,Sul


## Oferta: leitos SUS por 1.000 habitantes

Esta feature mede oferta local relativa ao tamanho da população. Valores baixos podem indicar baixa capacidade local, mas precisam ser lidos junto com fluxo regional, distância e demanda observada.

In [11]:
ano_referencia = int(features_named["ano"].max())
tipo_referencia = "uti_adulto"

cols_rank = [
    "municipio_nome",
    "ano",
    "tipo_leito",
    "populacao_total",
    "leitos_sus_por_1000_hab",
    "internacoes_por_1000_hab",
    "distancia_ate_polo_hospitalar",
    "saldo_pacientes_regional",
    "municipio_sem_leito_do_tipo",
]

(
    features_named
    .query("ano == @ano_referencia and tipo_leito == @tipo_referencia")
    .sort_values(["leitos_sus_por_1000_hab", "internacoes_por_1000_hab"], ascending=[True, False])
    [cols_rank]
    .head(20)
)

,municipio_nome,ano,tipo_leito,populacao_total,leitos_sus_por_1000_hab,internacoes_por_1000_hab,distancia_ate_polo_hospitalar,saldo_pacientes_regional,municipio_sem_leito_do_tipo
5389,Esperança do Sul,2025,uti_adulto,3293,0.0,16.398421,14.814778,-54,True
1609,Boa Vista do Cadeado,2025,uti_adulto,2269,0.0,13.662406,21.427846,-31,True
1679,Boa Vista do Sul,2025,uti_adulto,2815,0.0,13.143872,15.029846,-37,True
2834,Canudos do Vale,2025,uti_adulto,1686,0.0,12.455516,7.624127,-21,True
16904,Vespasiano Corrêa,2025,uti_adulto,1851,0.0,12.425716,9.276017,-23,True
12844,Santa Margarida do Sul,2025,uti_adulto,2661,0.0,12.401353,26.712625,-33,True
16064,Três Forquilhas,2025,uti_adulto,2813,0.0,12.08674,28.939612,-34,True
15749,Tiradentes do Sul,2025,uti_adulto,5188,0.0,11.950655,13.344726,-62,True
10009,Novo Cabrais,2025,uti_adulto,3633,0.0,11.835948,14.073955,-43,True
12984,Santa Tereza,2025,uti_adulto,1529,0.0,11.7724,10.484534,-18,True


## Demanda: internações e dias de permanência

`internacoes_por_1000_hab` mede volume de demanda de residentes. `dias_permanencia_por_1000_hab` mede carga assistencial, porque uma internação curta e uma internação longa não pressionam a rede da mesma forma.

In [12]:
(
    features_named
    .query("ano == @ano_referencia")
    .sort_values("dias_permanencia_por_1000_hab", ascending=False)
    [
        [
            "municipio_nome",
            "tipo_leito",
            "populacao_total",
            "internacoes_por_1000_hab",
            "dias_permanencia_por_1000_hab",
            "media_permanencia",
            "taxa_mortalidade_hospitalar",
        ]
    ]
    .head(20)
)

,municipio_nome,tipo_leito,populacao_total,internacoes_por_1000_hab,dias_permanencia_por_1000_hab,media_permanencia,taxa_mortalidade_hospitalar
11651,Progresso,leitos_clinicos,5423,111.008667,756.223493,6.812292,0.043189
14521,São Valentim do Sul,leitos_clinicos,2255,116.629712,701.995565,6.019011,0.030418
16901,Vespasiano Corrêa,leitos_clinicos,1851,139.384117,694.219341,4.980620,0.034884
11581,Pouso Novo,leitos_clinicos,1771,88.65048,666.290232,7.515924,0.050955
10076,Novo Machado,leitos_clinicos,3239,100.648348,622.414325,6.184049,0.064417
136,Ajuricaba,leitos_clinicos,6843,43.255882,621.803303,14.375000,0.118243
16236,Tucunduva,leitos_clinicos,5646,97.236982,617.251151,6.347905,0.029144
4021,Coqueiro Baixo,leitos_clinicos,1308,97.859327,601.681957,6.148438,0.046875
15046,Sertão,leitos_clinicos,5630,106.394316,590.941385,5.554257,0.038397
11196,Piratini,leitos_clinicos,17770,63.477772,560.720315,8.833333,0.078014


## Fluxo regional: saldo de pacientes

A leitura prática do `saldo_pacientes_regional` é:

- positivo: o município recebeu mais pacientes de fora do que enviou residentes para fora;
- negativo: o município enviou mais residentes para internação fora do que recebeu pacientes de fora.

Isso ajuda a separar municípios polo de municípios dependentes da rede regional.

In [13]:
saldo_cols = [
    "municipio_nome",
    "tipo_leito",
    "saldo_pacientes_regional",
    "internacoes_por_1000_hab",
    "leitos_sus_por_1000_hab",
    "distancia_ate_polo_hospitalar",
]

recebedores = (
    features_named
    .query("ano == @ano_referencia")
    .sort_values("saldo_pacientes_regional", ascending=False)
    [saldo_cols]
    .head(15)
)

dependentes = (
    features_named
    .query("ano == @ano_referencia")
    .sort_values("saldo_pacientes_regional", ascending=True)
    [saldo_cols]
    .head(15)
)

display(recebedores)
display(dependentes)

,municipio_nome,tipo_leito,saldo_pacientes_regional,internacoes_por_1000_hab,leitos_sus_por_1000_hab,distancia_ate_polo_hospitalar
11405,Porto Alegre,leitos_cirurgicos,27322,22.171035,0.798775,0.0
11406,Porto Alegre,leitos_clinicos,17926,30.498404,1.270407,0.0
11408,Porto Alegre,leitos_pediatricos,11945,9.041658,0.243917,0.0
10670,Passo Fundo,leitos_cirurgicos,8352,30.422092,0.856567,0.0
11409,Porto Alegre,uti_adulto,8307,7.218493,0.278659,0.0
11407,Porto Alegre,leitos_obstetricos,5229,8.588747,0.158891,0.0
15605,Tenente Portela,leitos_cirurgicos,4193,38.79892,2.699055,0.0
10671,Passo Fundo,leitos_clinicos,3709,30.026395,1.093209,0.0
5175,Erechim,leitos_cirurgicos,3687,32.999115,0.46529,0.0
10565,Parobé,leitos_cirurgicos,3594,47.172805,0.130673,0.0


,municipio_nome,tipo_leito,saldo_pacientes_regional,internacoes_por_1000_hab,leitos_sus_por_1000_hab,distancia_ate_polo_hospitalar
416,Alvorada,leitos_clinicos,-4021,30.572704,0.263661,0.00000
415,Alvorada,leitos_cirurgicos,-3849,22.966887,0.027053,0.00000
16970,Viamão,leitos_cirurgicos,-2843,19.017569,0.076151,0.00000
16971,Viamão,leitos_clinicos,-2580,26.397007,0.379317,0.00000
2165,Cachoeirinha,leitos_cirurgicos,-2195,18.988997,0.05418,0.00000
6505,Gravataí,leitos_cirurgicos,-2108,19.939731,0.058091,0.00000
2166,Cachoeirinha,leitos_clinicos,-1635,20.31759,0.197876,0.00000
6506,Gravataí,leitos_clinicos,-1523,14.355735,0.392114,0.00000
16973,Viamão,leitos_pediatricos,-1373,7.625131,0.050288,0.00000
5490,Estância Velha,leitos_cirurgicos,-1307,38.990687,0.176771,0.00000


## Distância até polo hospitalar

A distância aqui é uma aproximação geográfica até o município mais próximo com proxy de referência hospitalar no CNES. Ela é útil para triagem, mas não substitui tempo real de deslocamento, barreiras geográficas, regulação ou pactuação regional.

In [14]:
(
    features_named
    .query("ano == @ano_referencia")
    .drop_duplicates(["municipio_id", "ano"])
    .sort_values("distancia_ate_polo_hospitalar", ascending=False)
    [
        [
            "municipio_nome",
            "regiao_saude",
            "macroregiao_saude",
            "populacao_total",
            "distancia_ate_polo_hospitalar",
        ]
    ]
    .head(20)
)

,municipio_nome,regiao_saude,macroregiao_saude,populacao_total,distancia_ate_polo_hospitalar
15570,Tavares,05 - Bons Ventos,Metropolitana,5319,79.199624
1290,Barra do Quaraí,03 - Fronteira Oeste,Centro-Oeste,4340,67.814153
13820,São José do Norte,21- Região Sul,Sul,26248,67.060554
3740,Chuí,21- Região Sul,Sul,6409,66.197982
9235,Mostardas,05 - Bons Ventos,Metropolitana,12347,60.902061
7415,Itacurubi,02 - Entre Rios,Centro-Oeste,3038,52.059440
6155,Garruchos,11 - Sete Povos das Missões,Missioneira,2724,48.247729
14170,São Miguel das Missões,11 - Sete Povos das Missões,Missioneira,7193,45.836196
13190,Santo Antônio das Missões,11 - Sete Povos das Missões,Missioneira,10485,45.802402
13925,São José dos Ausentes,24 - Campos de Cima da Serra,Serra,4295,45.399838


## Tendência: crescimento em três anos

As features de crescimento comparam o ano atual contra `ano - 3` na mesma combinação `município + tipo_leito`.

Exemplo: `2024` é comparado com `2021`.

Isso evita comparar linhas vizinhas por acidente quando algum ano está ausente.

In [15]:
(
    features_named
    .query("ano == @ano_referencia")
    .sort_values("crescimento_internacoes_3anos", ascending=False, na_position="last")
    [
        [
            "municipio_nome",
            "tipo_leito",
            "internacoes_por_1000_hab",
            "crescimento_internacoes_3anos",
            "crescimento_populacao_3anos",
            "leitos_sus_por_1000_hab",
        ]
    ]
    .head(20)
)

,municipio_nome,tipo_leito,internacoes_por_1000_hab,crescimento_internacoes_3anos,crescimento_populacao_3anos,leitos_sus_por_1000_hab
7977,Lagoa Bonita do Sul,leitos_obstetricos,5.256242,11.000000,0.014216,0.0
9763,Nova Pádua,leitos_pediatricos,5.023022,11.000000,0.019633,0.0
14872,Selbach,leitos_obstetricos,4.404443,10.500000,0.022518,0.574493
13334,Santo Expedito do Sul,uti_adulto,4.175365,9.000000,0.019583,0.0
9764,Nova Pádua,uti_adulto,3.767267,8.000000,0.019633,0.0
8431,Manoel Viana,leitos_clinicos,24.150398,7.350000,0.016762,0.0
8434,Manoel Viana,uti_adulto,7.953724,6.857143,0.016762,0.0
8049,Lagoa dos Três Cantos,uti_adulto,3.930376,6.000000,0.024741,0.0
8433,Manoel Viana,leitos_pediatricos,10.845987,5.818182,0.016762,0.0
16903,Vespasiano Corrêa,leitos_pediatricos,19.448947,3.500000,0.018152,0.0


## Ausência de leito do tipo

Esta feature é booleana e precisa de cuidado. Ela diz apenas que, no ano observado, não houve leito SUS do tipo no município.

Boa leitura:

> Município sem oferta local do tipo, com possível dependência de atendimento regional.

Má leitura:

> Município sem leito, portanto deve receber leito.


In [16]:
features_named.groupby(["ano", "tipo_leito"]).agg(
    municipios_sem_leito=("municipio_sem_leito_do_tipo", "sum"),
    municipios_total=("municipio_id", "nunique"),
).assign(
    pct_sem_leito=lambda df: df["municipios_sem_leito"] / df["municipios_total"]
).reset_index().sort_values(["ano", "pct_sem_leito"], ascending=[True, False])

,ano,tipo_leito,municipios_sem_leito,municipios_total,pct_sem_leito
4,2019,uti_adulto,449,497,0.903421
0,2019,leitos_cirurgicos,289,497,0.581489
2,2019,leitos_obstetricos,285,497,0.573441
3,2019,leitos_pediatricos,271,497,0.545272
1,2019,leitos_clinicos,259,497,0.521127
9,2020,uti_adulto,434,497,0.873239
5,2020,leitos_cirurgicos,294,497,0.591549
7,2020,leitos_obstetricos,292,497,0.587525
8,2020,leitos_pediatricos,276,497,0.555332
6,2020,leitos_clinicos,264,497,0.531187


## Visão por tipo de leito

A célula abaixo cria uma pequena função para filtrar um tipo de leito e ano. Ela é útil para navegar sem repetir código.

In [17]:
def mostrar_tipo_leito(tipo_leito: str, ano: int | None = None, n: int = 15) -> pd.DataFrame:
    ano = ano or int(features_named["ano"].max())
    recorte = features_named.query("tipo_leito == @tipo_leito and ano == @ano").copy()
    recorte["sinal_demanda_com_baixa_oferta"] = (
        recorte["internacoes_por_1000_hab"].rank(pct=True)
        - recorte["leitos_sus_por_1000_hab"].rank(pct=True)
    )
    return (
        recorte.sort_values("sinal_demanda_com_baixa_oferta", ascending=False)
        [
            [
                "municipio_nome",
                "regiao_saude",
                "populacao_total",
                "leitos_sus_por_1000_hab",
                "internacoes_por_1000_hab",
                "dias_permanencia_por_1000_hab",
                "saldo_pacientes_regional",
                "distancia_ate_polo_hospitalar",
                "municipio_sem_leito_do_tipo",
                "sinal_demanda_com_baixa_oferta",
            ]
        ]
        .head(n)
    )


mostrar_tipo_leito("leitos_clinicos", ano_referencia)

,municipio_nome,regiao_saude,populacao_total,leitos_sus_por_1000_hab,internacoes_por_1000_hab,dias_permanencia_por_1000_hab,saldo_pacientes_regional,distancia_ate_polo_hospitalar,municipio_sem_leito_do_tipo,sinal_demanda_com_baixa_oferta
16901,Vespasiano Corrêa,29 - Vales e Montanhas,1851,0.0,139.384117,694.219341,-258,9.276017,True,0.737425
14521,São Valentim do Sul,29 - Vales e Montanhas,2255,0.0,116.629712,701.995565,-263,10.703133,True,0.735412
10076,Novo Machado,14 - Fronteira Noroeste,3239,0.0,100.648348,622.414325,-326,14.834875,True,0.723340
4021,Coqueiro Baixo,29 - Vales e Montanhas,1308,0.0,97.859327,601.681957,-128,13.648682,True,0.719316
11581,Pouso Novo,29 - Vales e Montanhas,1771,0.0,88.65048,666.290232,-157,11.449814,True,0.711268
6436,Gramado dos Loureiros,20 - Rota da Produção,2047,0.0,81.094284,423.546654,-166,9.257539,True,0.703219
7241,Inhacorá,13 - Diversidade,2047,0.0,76.697606,449.926722,-157,15.092636,True,0.693159
8816,Mato Queimado,11 - Sete Povos das Missões,1833,0.0,75.286416,373.70431,-138,10.810785,True,0.691147
7976,Lagoa Bonita do Sul,27 - Jacuí Centro,2283,0.0,73.149365,335.523434,-167,10.885747,True,0.683099
15886,Travesseiro,29 - Vales e Montanhas,2192,0.0,72.536496,338.50365,-159,5.983323,True,0.681087


In [18]:
for tipo in sorted(features_named["tipo_leito"].unique()):
    print(f"\n=== {tipo} ===")
    display(mostrar_tipo_leito(tipo, ano_referencia, n=10))


=== leitos_cirurgicos ===


,municipio_nome,regiao_saude,populacao_total,leitos_sus_por_1000_hab,internacoes_por_1000_hab,dias_permanencia_por_1000_hab,saldo_pacientes_regional,distancia_ate_polo_hospitalar,municipio_sem_leito_do_tipo,sinal_demanda_com_baixa_oferta
10950,Pinhal,15 - Caminho das águas,3039,0.0,90.161237,194.800921,-274,9.671701,True,0.689135
12980,Santa Tereza,25 - Vinhedos e Basalto,1529,0.0,69.980379,81.098757,-107,10.484534,True,0.687123
4510,Derrubadas,15 - Caminho das águas,2793,0.0,66.237021,100.966702,-185,19.488539,True,0.685111
4335,Cristal do Sul,15 - Caminho das águas,2745,0.0,64.845173,112.932605,-178,8.426266,True,0.683099
10110,Novo Tiradentes,15 - Caminho das águas,2187,0.0,64.471879,126.657522,-141,10.949278,True,0.681087
11335,Ponte Preta,16 - Alto Uruguai Gaúcho,1602,0.0,61.173533,64.918851,-98,15.205398,True,0.679074
17215,Vista Alegre,15 - Caminho das águas,2710,0.0,60.516605,90.774908,-164,7.668450,True,0.675050
1675,Boa Vista do Sul,25 - Vinhedos e Basalto,2815,0.0,60.390764,65.008881,-170,15.029846,True,0.673038
8990,Monte Alegre dos Campos,24 - Campos de Cima da Serra,3251,0.0,56.905568,93.817287,-185,38.467277,True,0.669014
4160,Coronel Pilar,25 - Vinhedos e Basalto,1637,0.0,56.81124,35.430666,-93,10.585563,True,0.667002



=== leitos_clinicos ===


,municipio_nome,regiao_saude,populacao_total,leitos_sus_por_1000_hab,internacoes_por_1000_hab,dias_permanencia_por_1000_hab,saldo_pacientes_regional,distancia_ate_polo_hospitalar,municipio_sem_leito_do_tipo,sinal_demanda_com_baixa_oferta
16901,Vespasiano Corrêa,29 - Vales e Montanhas,1851,0.0,139.384117,694.219341,-258,9.276017,True,0.737425
14521,São Valentim do Sul,29 - Vales e Montanhas,2255,0.0,116.629712,701.995565,-263,10.703133,True,0.735412
10076,Novo Machado,14 - Fronteira Noroeste,3239,0.0,100.648348,622.414325,-326,14.834875,True,0.723340
4021,Coqueiro Baixo,29 - Vales e Montanhas,1308,0.0,97.859327,601.681957,-128,13.648682,True,0.719316
11581,Pouso Novo,29 - Vales e Montanhas,1771,0.0,88.65048,666.290232,-157,11.449814,True,0.711268
6436,Gramado dos Loureiros,20 - Rota da Produção,2047,0.0,81.094284,423.546654,-166,9.257539,True,0.703219
7241,Inhacorá,13 - Diversidade,2047,0.0,76.697606,449.926722,-157,15.092636,True,0.693159
8816,Mato Queimado,11 - Sete Povos das Missões,1833,0.0,75.286416,373.70431,-138,10.810785,True,0.691147
7976,Lagoa Bonita do Sul,27 - Jacuí Centro,2283,0.0,73.149365,335.523434,-167,10.885747,True,0.683099
15886,Travesseiro,29 - Vales e Montanhas,2192,0.0,72.536496,338.50365,-159,5.983323,True,0.681087



=== leitos_obstetricos ===


,municipio_nome,regiao_saude,populacao_total,leitos_sus_por_1000_hab,internacoes_por_1000_hab,dias_permanencia_por_1000_hab,saldo_pacientes_regional,distancia_ate_polo_hospitalar,municipio_sem_leito_do_tipo,sinal_demanda_com_baixa_oferta
3672,Charrua,16 - Alto Uruguai Gaúcho,2800,0.0,22.5,53.571429,-63,12.662781,True,0.671026
14557,São Valério do Sul,13 - Diversidade,2593,0.0,20.439645,74.431161,-53,11.129778,True,0.669014
11897,Redentora,20 - Rota da Produção,9931,0.0,18.628537,48.836975,-185,21.920027,True,0.667002
6437,Gramado dos Loureiros,20 - Rota da Produção,2047,0.0,17.586712,29.799707,-36,9.257539,True,0.664990
1537,Boa Vista das Missões,20 - Rota da Produção,1968,0.0,17.276423,23.882114,-34,11.059372,True,0.662978
1782,Bom Progresso,15 - Caminho das águas,2132,0.0,16.885553,37.054409,-36,13.305716,True,0.660966
3497,Cerro Grande,20 - Rota da Produção,2428,0.0,15.650741,36.655684,-38,9.952993,True,0.658954
1467,Benjamin Constant do Sul,16 - Alto Uruguai Gaúcho,2118,0.0,15.580737,42.020774,-33,24.760631,True,0.656942
5142,Erebango,16 - Alto Uruguai Gaúcho,3122,0.0,15.37476,39.077514,-48,11.387049,True,0.652918
9657,Nova Candelária,14 - Fronteira Noroeste,3138,0.0,15.296367,43.021033,-48,9.199454,True,0.650905



=== leitos_pediatricos ===


,municipio_nome,regiao_saude,populacao_total,leitos_sus_por_1000_hab,internacoes_por_1000_hab,dias_permanencia_por_1000_hab,saldo_pacientes_regional,distancia_ate_polo_hospitalar,municipio_sem_leito_do_tipo,sinal_demanda_com_baixa_oferta
17218,Vista Alegre,15 - Caminho das águas,2710,0.0,29.151292,74.907749,-79,7.668450,True,0.706237
6438,Gramado dos Loureiros,20 - Rota da Produção,2047,0.0,28.822667,122.618466,-59,9.257539,True,0.704225
10953,Pinhal,15 - Caminho das águas,3039,0.0,27.640671,127.015466,-84,9.671701,True,0.702213
14558,São Valério do Sul,13 - Diversidade,2593,0.0,26.995758,138.064019,-70,11.129778,True,0.700201
11583,Pouso Novo,29 - Vales e Montanhas,1771,0.0,24.280068,130.999435,-43,11.449814,True,0.696177
5773,Faxinalzinho,16 - Alto Uruguai Gaúcho,2572,0.0,23.328149,59.875583,-60,17.890563,True,0.692153
12003,Rio dos Índios,16 - Alto Uruguai Gaúcho,2866,0.0,23.028611,57.571528,-66,13.161234,True,0.690141
1258,Barra do Guarita,15 - Caminho das águas,3231,0.0,22.593624,168.368926,-73,18.601310,True,0.686117
16098,Três Palmeiras,20 - Rota da Produção,4829,0.0,22.157797,132.739698,-107,11.651971,True,0.684105
11058,Pinheirinho do Vale,15 - Caminho das águas,4638,0.0,21.776628,101.552393,-101,12.662941,True,0.682093



=== uti_adulto ===


,municipio_nome,regiao_saude,populacao_total,leitos_sus_por_1000_hab,internacoes_por_1000_hab,dias_permanencia_por_1000_hab,saldo_pacientes_regional,distancia_ate_polo_hospitalar,municipio_sem_leito_do_tipo,sinal_demanda_com_baixa_oferta
5389,Esperança do Sul,15 - Caminho das águas,3293,0.0,16.398421,122.077133,-54,14.814778,True,0.556338
1609,Boa Vista do Cadeado,12 - Portal das Missões,2269,0.0,13.662406,171.000441,-31,21.427846,True,0.554326
1679,Boa Vista do Sul,25 - Vinhedos e Basalto,2815,0.0,13.143872,105.506217,-37,15.029846,True,0.552314
2834,Canudos do Vale,29 - Vales e Montanhas,1686,0.0,12.455516,145.314353,-21,7.624127,True,0.548290
16904,Vespasiano Corrêa,29 - Vales e Montanhas,1851,0.0,12.425716,119.93517,-23,9.276017,True,0.546278
12844,Santa Margarida do Sul,03 - Fronteira Oeste,2661,0.0,12.401353,194.66366,-33,26.712625,True,0.544266
16064,Três Forquilhas,04 - Belas Praias,2813,0.0,12.08674,125.844294,-34,28.939612,True,0.542254
15749,Tiradentes do Sul,15 - Caminho das águas,5188,0.0,11.950655,78.835775,-62,13.344726,True,0.540241
10009,Novo Cabrais,27 - Jacuí Centro,3633,0.0,11.835948,148.912744,-43,14.073955,True,0.538229
12984,Santa Tereza,25 - Vinhedos e Basalto,1529,0.0,11.7724,94.179202,-18,10.484534,True,0.536217


## Features específicas: leitos clínicos

A primeira tabela específica implementada é `features_leitos_clinicos.parquet`.

Ela mantém a unidade `município + ano`, mas deixa de ser genérica: cada coluna foi escolhida para avaliar pressão sobre leitos clínicos SUS, combinando demanda clínica, permanência, envelhecimento populacional, mortalidade por grupos relevantes e oferta local.

In [ ]:
features_clinicos.shape

In [ ]:
features_clinicos.head(10)

In [ ]:
clinical_catalog = pd.DataFrame(
    [
        {
            "feature": "internacoes_clinicas_12m",
            "grupo": "demanda",
            "interpretação": "Total anual de internações clínicas de residentes do município.",
            "fonte": "SIH/SUS",
            "cuidado": "Classificação clínica é proxy baseada no grupo de procedimento do SIH.",
        },
        {
            "feature": "internacoes_clinicas_por_1000_hab",
            "grupo": "demanda relativa",
            "interpretação": "Internações clínicas anuais por 1.000 habitantes.",
            "fonte": "SIH/SUS + IBGE",
            "cuidado": "Ajuda a comparar municípios de tamanhos diferentes.",
        },
        {
            "feature": "dias_clinicos_permanencia",
            "grupo": "pressão assistencial",
            "interpretação": "Soma anual dos dias de permanência em internações clínicas.",
            "fonte": "SIH/SUS",
            "cuidado": "Pode refletir volume, gravidade, permanência prolongada ou dificuldade de alta.",
        },
        {
            "feature": "populacao_60_mais",
            "grupo": "perfil populacional",
            "interpretação": "População de 60 anos ou mais no município.",
            "fonte": "IBGE",
            "cuidado": "Estimativa proporcional pela estrutura idade/sexo disponível.",
        },
        {
            "feature": "percentual_idosos",
            "grupo": "perfil populacional",
            "interpretação": "Participação da população 60+ na população total.",
            "fonte": "IBGE",
            "cuidado": "Sinal importante para leitos clínicos, mas não é demanda hospitalar direta.",
        },
        {
            "feature": "mortalidade_respiratoria",
            "grupo": "risco",
            "interpretação": "Óbitos respiratórios por 1.000 habitantes.",
            "fonte": "SIM + IBGE",
            "cuidado": "Não indica causalidade nem qualidade da rede local.",
        },
        {
            "feature": "mortalidade_cardiovascular",
            "grupo": "risco",
            "interpretação": "Óbitos cardiovasculares por 1.000 habitantes.",
            "fonte": "SIM + IBGE",
            "cuidado": "Útil como sinal complementar de vulnerabilidade clínica.",
        },
        {
            "feature": "leitos_clinicos_por_1000_hab",
            "grupo": "oferta",
            "interpretação": "Média anual de leitos clínicos SUS por 1.000 habitantes.",
            "fonte": "CNES + IBGE",
            "cuidado": "Usa QT_SUS médio mensal; não mede ocupação nem disponibilidade operacional diária.",
        },
    ]
)

clinical_catalog

### Cobertura e nulos

A tabela clínica tem uma linha por `município + ano`. As mortalidades respiratória e cardiovascular podem ficar ausentes em anos sem SIM processado, especialmente `2025` no estado atual do workspace. Isso é melhor do que preencher zero, porque zero indicaria ausência real de óbitos.

In [ ]:
clinical_coverage = {
    "linhas": len(features_clinicos),
    "municipios": features_clinicos["municipio_id"].nunique(),
    "anos": sorted(features_clinicos["ano"].unique().tolist()),
    "chave_duplicada": features_clinicos.duplicated(["municipio_id", "ano"]).any(),
}

clinical_coverage

In [ ]:
(
    features_clinicos.isna()
    .sum()
    .rename("valores_ausentes")
    .to_frame()
    .assign(percentual=lambda df: df["valores_ausentes"] / len(features_clinicos))
)

### Leitura exploratória

A célula abaixo junta nomes e regiões para facilitar a leitura. O ranking é apenas exploratório: ele combina demanda clínica e envelhecimento com baixa oferta relativa, mas ainda não é o score final do projeto.

In [ ]:
features_clinicos_named = features_clinicos.merge(
    dim[["municipio_id", "municipio_nome", "regiao_saude", "macroregiao_saude"]],
    on="municipio_id",
    how="left",
    validate="many_to_one",
)

features_clinicos_named.head()

In [ ]:
features_clinicos_named.groupby("ano").agg(
    municipios=("municipio_id", "nunique"),
    internacoes_clinicas=("internacoes_clinicas_12m", "sum"),
    dias_clinicos=("dias_clinicos_permanencia", "sum"),
    media_internacoes_1000=("internacoes_clinicas_por_1000_hab", "mean"),
    media_percentual_idosos=("percentual_idosos", "mean"),
    media_leitos_1000=("leitos_clinicos_por_1000_hab", "mean"),
)

In [ ]:
ano_clinico = int(features_clinicos_named["ano"].max())

(
    features_clinicos_named.query("ano == @ano_clinico")
    .assign(
        sinal_clinico_exploratorio=lambda df: df["internacoes_clinicas_por_1000_hab"].rank(pct=True)
        + df["percentual_idosos"].rank(pct=True)
        - df["leitos_clinicos_por_1000_hab"].rank(pct=True)
    )
    .sort_values("sinal_clinico_exploratorio", ascending=False)
    [
        [
            "municipio_nome",
            "regiao_saude",
            "internacoes_clinicas_12m",
            "internacoes_clinicas_por_1000_hab",
            "dias_clinicos_permanencia",
            "percentual_idosos",
            "mortalidade_respiratoria",
            "mortalidade_cardiovascular",
            "leitos_clinicos_por_1000_hab",
            "sinal_clinico_exploratorio",
        ]
    ]
    .head(20)
)

## Features específicas: leitos cirúrgicos

A tabela `features_leitos_cirurgicos.parquet` concentra os sinais próprios da análise de leitos cirúrgicos SUS.

A leitura aqui combina volume de internações/procedimentos cirúrgicos, tempo médio de permanência, valor médio da AIH, oferta local de leitos cirúrgicos e existência de estrutura física de centro cirúrgico.

In [ ]:
features_cirurgicos.shape

In [ ]:
features_cirurgicos.head(10)

In [ ]:
surgical_catalog = pd.DataFrame(
    [
        {
            "feature": "internacoes_cirurgicas_12m",
            "grupo": "demanda",
            "interpretação": "Total anual de internações cirúrgicas de residentes do município.",
            "fonte": "SIH/SUS",
            "cuidado": "Classificação cirúrgica é proxy baseada no grupo de procedimento do SIH.",
        },
        {
            "feature": "procedimentos_cirurgicos_12m",
            "grupo": "demanda",
            "interpretação": "Proxy atual para procedimentos cirúrgicos hospitalares no ano.",
            "fonte": "SIH/SUS",
            "cuidado": "Neste MVP inicial é igual às internações cirúrgicas classificadas; pode ser refinado por SIGTAP exato depois.",
        },
        {
            "feature": "tempo_medio_permanencia_cirurgico",
            "grupo": "pressão assistencial",
            "interpretação": "Média de dias de permanência das internações cirúrgicas.",
            "fonte": "SIH/SUS",
            "cuidado": "Valores altos podem indicar complexidade, permanência prolongada ou perfil específico dos procedimentos.",
        },
        {
            "feature": "valor_medio_cirurgico",
            "grupo": "custo/proxy de complexidade",
            "interpretação": "Valor médio da AIH nas internações cirúrgicas.",
            "fonte": "SIH/SUS",
            "cuidado": "Não mede custo real total, mas ajuda a sinalizar complexidade relativa do atendimento registrado.",
        },
        {
            "feature": "leitos_cirurgicos_por_1000_hab",
            "grupo": "oferta",
            "interpretação": "Média anual de leitos cirúrgicos SUS por 1.000 habitantes.",
            "fonte": "CNES + IBGE",
            "cuidado": "Usa QT_SUS médio mensal; não mede ocupação nem fila cirúrgica reprimida.",
        },
        {
            "feature": "estabelecimentos_com_centro_cirurgico",
            "grupo": "estrutura",
            "interpretação": "Maior contagem mensal no ano de estabelecimentos com centro cirúrgico registrado.",
            "fonte": "CNES ST",
            "cuidado": "É proxy de estrutura física, não garantia de equipe, escala, habilitação ou capacidade operacional plena.",
        },
    ]
)

surgical_catalog

### Cobertura e nulos cirúrgicos

A tabela cirúrgica também tem uma linha por `município + ano`. Como as features usam CNES, SIH, IBGE e CNES ST, não há nulos esperados quando essas facts estão presentes no período `2019-2025`.

In [ ]:
surgical_coverage = {
    "linhas": len(features_cirurgicos),
    "municipios": features_cirurgicos["municipio_id"].nunique(),
    "anos": sorted(features_cirurgicos["ano"].unique().tolist()),
    "chave_duplicada": features_cirurgicos.duplicated(["municipio_id", "ano"]).any(),
}

surgical_coverage

In [ ]:
(
    features_cirurgicos.isna()
    .sum()
    .rename("valores_ausentes")
    .to_frame()
    .assign(percentual=lambda df: df["valores_ausentes"] / len(features_cirurgicos))
)

### Leitura exploratória cirúrgica

O ranking abaixo é exploratório. Ele combina alta demanda cirúrgica, maior permanência média, existência de centro cirúrgico e menor oferta relativa de leitos. Ainda não é o score final de priorização.

In [ ]:
features_cirurgicos_named = features_cirurgicos.merge(
    dim[["municipio_id", "municipio_nome", "regiao_saude", "macroregiao_saude"]],
    on="municipio_id",
    how="left",
    validate="many_to_one",
)

features_cirurgicos_named.head()

In [ ]:
features_cirurgicos_named.groupby("ano").agg(
    municipios=("municipio_id", "nunique"),
    internacoes_cirurgicas=("internacoes_cirurgicas_12m", "sum"),
    procedimentos_cirurgicos=("procedimentos_cirurgicos_12m", "sum"),
    media_permanencia=("tempo_medio_permanencia_cirurgico", "mean"),
    valor_medio=("valor_medio_cirurgico", "mean"),
    media_leitos_1000=("leitos_cirurgicos_por_1000_hab", "mean"),
    municipios_com_centro_cirurgico=("estabelecimentos_com_centro_cirurgico", lambda values: values.gt(0).sum()),
)

In [ ]:
ano_cirurgico = int(features_cirurgicos_named["ano"].max())

(
    features_cirurgicos_named.query("ano == @ano_cirurgico")
    .assign(
        sinal_cirurgico_exploratorio=lambda df: df["internacoes_cirurgicas_12m"].rank(pct=True)
        + df["tempo_medio_permanencia_cirurgico"].rank(pct=True)
        + df["estabelecimentos_com_centro_cirurgico"].gt(0).astype(int) * 0.25
        - df["leitos_cirurgicos_por_1000_hab"].rank(pct=True)
    )
    .sort_values("sinal_cirurgico_exploratorio", ascending=False)
    [
        [
            "municipio_nome",
            "regiao_saude",
            "internacoes_cirurgicas_12m",
            "procedimentos_cirurgicos_12m",
            "tempo_medio_permanencia_cirurgico",
            "valor_medio_cirurgico",
            "leitos_cirurgicos_por_1000_hab",
            "estabelecimentos_com_centro_cirurgico",
            "sinal_cirurgico_exploratorio",
        ]
    ]
    .head(20)
)

## Features específicas: leitos obstétricos

A tabela `features_leitos_obstetricos.parquet` concentra os sinais próprios da análise de leitos obstétricos SUS.

A leitura aqui combina nascimentos, partos, população feminina em idade reprodutiva, oferta obstétrica SUS e distância até uma proxy de maternidade de referência.

In [ ]:
features_obstetricos.shape

In [ ]:
features_obstetricos.head(10)

In [ ]:
obstetric_catalog = pd.DataFrame(
    [
        {
            "feature": "nascidos_vivos_12m",
            "grupo": "demanda",
            "interpretação": "Nascidos vivos no ano por município de residência da mãe.",
            "fonte": "SINASC",
            "cuidado": "Não é internação hospitalar, mas é um sinal direto de demanda obstétrica territorial.",
        },
        {
            "feature": "nascidos_vivos_por_1000_hab",
            "grupo": "demanda relativa",
            "interpretação": "Nascidos vivos por 1.000 habitantes.",
            "fonte": "SINASC + IBGE",
            "cuidado": "Útil para comparação, mas população total não substitui denominadores obstétricos específicos.",
        },
        {
            "feature": "mulheres_10_49",
            "grupo": "denominador",
            "interpretação": "Mulheres de 10 a 49 anos estimadas para o município.",
            "fonte": "IBGE",
            "cuidado": "Estimativa proporcional pela estrutura idade/sexo disponível.",
        },
        {
            "feature": "partos_12m",
            "grupo": "demanda",
            "interpretação": "Soma anual de partos cesáreos e vaginais.",
            "fonte": "SINASC",
            "cuidado": "Reflete eventos de nascimento registrados, não necessariamente uso de leito obstétrico no município de residência.",
        },
        {
            "feature": "partos_cesareos",
            "grupo": "demanda/perfil assistencial",
            "interpretação": "Total anual de partos cesáreos.",
            "fonte": "SINASC",
            "cuidado": "A proporção cesárea/vaginal precisa ser interpretada com contexto assistencial.",
        },
        {
            "feature": "partos_vaginais",
            "grupo": "demanda/perfil assistencial",
            "interpretação": "Total anual de partos vaginais.",
            "fonte": "SINASC",
            "cuidado": "Complementa partos cesáreos para descrever o perfil obstétrico registrado.",
        },
        {
            "feature": "leitos_obstetricos_por_1000_mulheres_10_49",
            "grupo": "oferta",
            "interpretação": "Média anual de leitos obstétricos SUS por 1.000 mulheres de 10 a 49 anos.",
            "fonte": "CNES + IBGE",
            "cuidado": "Usa QT_SUS médio mensal; não mede ocupação nem qualidade da assistência obstétrica.",
        },
        {
            "feature": "distancia_ate_maternidade_referencia",
            "grupo": "acesso espacial",
            "interpretação": "Distância em km até município com proxy de estrutura de maternidade.",
            "fonte": "CNES ST + dimensão municipal",
            "cuidado": "É distância geográfica em linha reta; não substitui tempo de deslocamento, regulação ou rede pactuada.",
        },
    ]
)

obstetric_catalog

### Cobertura e nulos obstétricos

A tabela obstétrica tem uma linha por `município + ano`. As features derivadas do SINASC podem ficar nulas em `2025`, porque a base complementar disponível no workspace cobre `2019-2024`.

In [ ]:
obstetric_coverage = {
    "linhas": len(features_obstetricos),
    "municipios": features_obstetricos["municipio_id"].nunique(),
    "anos": sorted(features_obstetricos["ano"].unique().tolist()),
    "chave_duplicada": features_obstetricos.duplicated(["municipio_id", "ano"]).any(),
}

obstetric_coverage

In [ ]:
(
    features_obstetricos.isna()
    .sum()
    .rename("valores_ausentes")
    .to_frame()
    .assign(percentual=lambda df: df["valores_ausentes"] / len(features_obstetricos))
)

### Leitura exploratória obstétrica

O ranking abaixo é exploratório. Ele combina maior volume de nascimentos, maior distância até maternidade de referência e menor oferta relativa de leitos obstétricos. Ainda não é o score final de priorização.

In [ ]:
features_obstetricos_named = features_obstetricos.merge(
    dim[["municipio_id", "municipio_nome", "regiao_saude", "macroregiao_saude"]],
    on="municipio_id",
    how="left",
    validate="many_to_one",
)

features_obstetricos_named.head()

In [ ]:
features_obstetricos_named.groupby("ano").agg(
    municipios=("municipio_id", "nunique"),
    nascidos_vivos=("nascidos_vivos_12m", "sum"),
    partos=("partos_12m", "sum"),
    mulheres_10_49=("mulheres_10_49", "sum"),
    media_nascidos_1000=("nascidos_vivos_por_1000_hab", "mean"),
    media_leitos_mulheres_1000=("leitos_obstetricos_por_1000_mulheres_10_49", "mean"),
    distancia_media_maternidade=("distancia_ate_maternidade_referencia", "mean"),
)

In [ ]:
ano_obstetrico = int(features_obstetricos_named["ano"].max())

(
    features_obstetricos_named.query("ano == @ano_obstetrico")
    .assign(
        sinal_obstetrico_exploratorio=lambda df: df["nascidos_vivos_12m"].rank(pct=True)
        + df["distancia_ate_maternidade_referencia"].rank(pct=True)
        - df["leitos_obstetricos_por_1000_mulheres_10_49"].rank(pct=True)
    )
    .sort_values("sinal_obstetrico_exploratorio", ascending=False)
    [
        [
            "municipio_nome",
            "regiao_saude",
            "nascidos_vivos_12m",
            "nascidos_vivos_por_1000_hab",
            "mulheres_10_49",
            "partos_12m",
            "partos_cesareos",
            "partos_vaginais",
            "leitos_obstetricos_por_1000_mulheres_10_49",
            "distancia_ate_maternidade_referencia",
            "sinal_obstetrico_exploratorio",
        ]
    ]
    .head(20)
)

## Features específicas: UTI Adulto

A tabela `features_uti_adulto.parquet` concentra sinais próprios da análise de UTI Adulto SUS.

A leitura combina internações adultas com uso de UTI, dias de UTI, mortalidade hospitalar nessas internações, causas cardiovasculares e respiratórias, população idosa e oferta SUS de UTI Adulto. Esses sinais continuam sendo apoio para investigação, não regra automática de investimento.

In [ ]:
features_uti_adulto.shape

In [ ]:
features_uti_adulto.head(10)

In [ ]:
uti_adulto_catalog = pd.DataFrame(
    [
        {
            "feature": "internacoes_com_uti_adulto",
            "grupo": "demanda intensiva",
            "interpretação": "Total anual de internações de residentes adultos com uso registrado de UTI.",
            "fonte": "SIH",
            "cuidado": "Usa idade maior ou igual a 18 anos e flag de uso de UTI; não identifica diretamente o tipo físico do leito ocupado.",
        },
        {
            "feature": "populacao_60_mais",
            "grupo": "vulnerabilidade/denominador",
            "interpretação": "População estimada de 60 anos ou mais no município.",
            "fonte": "IBGE",
            "cuidado": "Estimativa proporcional pela estrutura idade/sexo disponível; não é o denominador direto da taxa de leitos.",
        },
        {
            "feature": "mortalidade_hospitalar_adulta",
            "grupo": "gravidade/desfecho",
            "interpretação": "Proporção de internações adultas com UTI que terminaram em óbito hospitalar.",
            "fonte": "SIH",
            "cuidado": "É mortalidade hospitalar entre internações com uso de UTI, não mortalidade populacional nem qualidade assistencial isolada.",
        },
        {
            "feature": "internacoes_cardiovasculares",
            "grupo": "perfil de demanda",
            "interpretação": "Total anual de internações adultas com diagnóstico principal CID-10 iniciado por I.",
            "fonte": "SIH",
            "cuidado": "Sinal amplo de carga cardiovascular; não significa automaticamente necessidade de UTI ou UTI coronariana.",
        },
        {
            "feature": "internacoes_respiratorias",
            "grupo": "perfil de demanda",
            "interpretação": "Total anual de internações adultas com diagnóstico principal CID-10 iniciado por J.",
            "fonte": "SIH",
            "cuidado": "Sinal amplo de carga respiratória; deve ser lido com sazonalidade, surtos e rede regional.",
        },
        {
            "feature": "dias_uti_adulto",
            "grupo": "carga assistencial intensiva",
            "interpretação": "Soma anual de dias de UTI em internações adultas com uso de UTI.",
            "fonte": "SIH",
            "cuidado": "Soma `dias_uti_mes` e `dias_uti_intercorrencia`; depende da qualidade de preenchimento do SIH.",
        },
        {
            "feature": "uti_adulto_por_1000_adultos",
            "grupo": "oferta",
            "interpretação": "Média anual de leitos SUS de UTI Adulto por 1.000 adultos proxy.",
            "fonte": "CNES + IBGE",
            "cuidado": "Denominador adulto usa `populacao_total - pop_0_14` como aproximação; usa QT_SUS médio mensal.",
        },
    ]
)

uti_adulto_catalog

### Cobertura e nulos UTI Adulto

A tabela de UTI Adulto tem uma linha por `município + ano`. Em anos sem internação adulta com uso de UTI, a mortalidade hospitalar adulta fica nula porque não há denominador de internações.

In [ ]:
uti_adulto_coverage = {
    "linhas": len(features_uti_adulto),
    "municipios": features_uti_adulto["municipio_id"].nunique(),
    "anos": sorted(features_uti_adulto["ano"].unique().tolist()),
    "chave_duplicada": features_uti_adulto.duplicated(["municipio_id", "ano"]).any(),
}

uti_adulto_coverage

In [ ]:
(
    features_uti_adulto.isna()
    .sum()
    .rename("valores_ausentes")
    .to_frame()
    .assign(percentual=lambda df: df["valores_ausentes"] / len(features_uti_adulto))
)

### Leitura exploratória UTI Adulto

O ranking abaixo é exploratório. Ele combina maior uso de UTI, maior carga de dias de UTI, maior população idosa e menor oferta relativa de UTI Adulto SUS. Ainda não é o score final de priorização.

In [ ]:
features_uti_adulto_named = features_uti_adulto.merge(
    dim[["municipio_id", "municipio_nome", "regiao_saude", "macroregiao_saude"]],
    on="municipio_id",
    how="left",
    validate="many_to_one",
)

features_uti_adulto_named.head()

In [ ]:
features_uti_adulto_named.groupby("ano").agg(
    municipios=("municipio_id", "nunique"),
    internacoes_com_uti_adulto=("internacoes_com_uti_adulto", "sum"),
    dias_uti_adulto=("dias_uti_adulto", "sum"),
    populacao_60_mais=("populacao_60_mais", "sum"),
    internacoes_cardiovasculares=("internacoes_cardiovasculares", "sum"),
    internacoes_respiratorias=("internacoes_respiratorias", "sum"),
    media_mortalidade_hospitalar=("mortalidade_hospitalar_adulta", "mean"),
    media_uti_adulto_1000_adultos=("uti_adulto_por_1000_adultos", "mean"),
)

In [ ]:
ano_uti_adulto = int(features_uti_adulto_named["ano"].max())

(
    features_uti_adulto_named.query("ano == @ano_uti_adulto")
    .assign(
        sinal_uti_adulto_exploratorio=lambda df: df["internacoes_com_uti_adulto"].rank(pct=True)
        + df["dias_uti_adulto"].rank(pct=True)
        + df["populacao_60_mais"].rank(pct=True)
        - df["uti_adulto_por_1000_adultos"].rank(pct=True)
    )
    .sort_values("sinal_uti_adulto_exploratorio", ascending=False)
    [
        [
            "municipio_nome",
            "regiao_saude",
            "internacoes_com_uti_adulto",
            "dias_uti_adulto",
            "populacao_60_mais",
            "mortalidade_hospitalar_adulta",
            "internacoes_cardiovasculares",
            "internacoes_respiratorias",
            "uti_adulto_por_1000_adultos",
            "sinal_uti_adulto_exploratorio",
        ]
    ]
    .head(20)
)

## Features específicas: UTI Pediátrica

A tabela `features_uti_pediatrica.parquet` concentra sinais próprios da análise de UTI Pediátrica SUS.

A leitura combina população infantil, internações pediátricas graves, uso de UTI em crianças, dias de UTI, mortalidade hospitalar pediátrica nas internações com UTI e oferta SUS de UTI Pediátrica. Esses sinais ajudam a levantar prioridades relativas para investigação, sem transformar ausência local de leito em decisão automática.

In [ ]:
features_uti_pediatrica.shape

In [ ]:
features_uti_pediatrica.head(10)

In [ ]:
uti_pediatrica_catalog = pd.DataFrame(
    [
        {
            "feature": "populacao_0_14",
            "grupo": "denominador",
            "interpretação": "População estimada de 0 a 14 anos no município.",
            "fonte": "IBGE",
            "cuidado": "Estimativa proporcional pela estrutura idade/sexo disponível; aproxima a população infantil do MVP.",
        },
        {
            "feature": "internacoes_pediatricas_graves",
            "grupo": "demanda grave",
            "interpretação": "Total anual de internações pediátricas com uso de UTI ou óbito hospitalar.",
            "fonte": "SIH",
            "cuidado": "É uma proxy transparente de gravidade; não substitui escore clínico nem regulação assistencial.",
        },
        {
            "feature": "internacoes_pediatricas_com_uti",
            "grupo": "demanda intensiva",
            "interpretação": "Total anual de internações de residentes de 0 a 14 anos com uso registrado de UTI.",
            "fonte": "SIH",
            "cuidado": "SIH registra uso de UTI, mas não identifica diretamente o tipo físico do leito ocupado.",
        },
        {
            "feature": "dias_uti_pediatrica",
            "grupo": "carga assistencial intensiva",
            "interpretação": "Soma anual de dias de UTI em internações pediátricas com uso de UTI.",
            "fonte": "SIH",
            "cuidado": "Soma `dias_uti_mes` e `dias_uti_intercorrencia`; depende da qualidade de preenchimento do SIH.",
        },
        {
            "feature": "mortalidade_pediatrica_hospitalar",
            "grupo": "gravidade/desfecho",
            "interpretação": "Proporção de internações pediátricas com UTI que terminaram em óbito hospitalar.",
            "fonte": "SIH",
            "cuidado": "É mortalidade hospitalar no subconjunto com UTI, não mortalidade infantil populacional nem medida isolada de qualidade.",
        },
        {
            "feature": "uti_pediatrica_por_1000_criancas",
            "grupo": "oferta",
            "interpretação": "Média anual de leitos SUS de UTI Pediátrica por 1.000 crianças de 0 a 14 anos.",
            "fonte": "CNES + IBGE",
            "cuidado": "Usa QT_SUS médio mensal e códigos CNES mapeados como UTI Pediátrica; não mede ocupação ou disponibilidade regulada.",
        },
    ]
)

uti_pediatrica_catalog

### Cobertura e nulos UTI Pediátrica

A tabela de UTI Pediátrica tem uma linha por `município + ano`. Em anos sem internação pediátrica com uso de UTI, a mortalidade pediátrica hospitalar fica nula porque não há denominador de internações com UTI.

In [ ]:
uti_pediatrica_coverage = {
    "linhas": len(features_uti_pediatrica),
    "municipios": features_uti_pediatrica["municipio_id"].nunique(),
    "anos": sorted(features_uti_pediatrica["ano"].unique().tolist()),
    "chave_duplicada": features_uti_pediatrica.duplicated(["municipio_id", "ano"]).any(),
}

uti_pediatrica_coverage

In [ ]:
(
    features_uti_pediatrica.isna()
    .sum()
    .rename("valores_ausentes")
    .to_frame()
    .assign(percentual=lambda df: df["valores_ausentes"] / len(features_uti_pediatrica))
)

### Leitura exploratória UTI Pediátrica

O ranking abaixo é exploratório. Ele combina maior uso pediátrico de UTI, maior carga de dias de UTI, maior população infantil, maior sinal de gravidade e menor oferta relativa de UTI Pediátrica SUS. Ainda não é o score final de priorização.

In [ ]:
features_uti_pediatrica_named = features_uti_pediatrica.merge(
    dim[["municipio_id", "municipio_nome", "regiao_saude", "macroregiao_saude"]],
    on="municipio_id",
    how="left",
    validate="many_to_one",
)

features_uti_pediatrica_named.head()

In [ ]:
features_uti_pediatrica_named.groupby("ano").agg(
    municipios=("municipio_id", "nunique"),
    internacoes_pediatricas_graves=("internacoes_pediatricas_graves", "sum"),
    internacoes_pediatricas_com_uti=("internacoes_pediatricas_com_uti", "sum"),
    dias_uti_pediatrica=("dias_uti_pediatrica", "sum"),
    populacao_0_14=("populacao_0_14", "sum"),
    media_mortalidade_pediatrica=("mortalidade_pediatrica_hospitalar", "mean"),
    media_uti_pediatrica_1000_criancas=("uti_pediatrica_por_1000_criancas", "mean"),
)

In [ ]:
ano_uti_pediatrica = int(features_uti_pediatrica_named["ano"].max())

(
    features_uti_pediatrica_named.query("ano == @ano_uti_pediatrica")
    .assign(
        sinal_uti_pediatrica_exploratorio=lambda df: df["internacoes_pediatricas_com_uti"].rank(pct=True)
        + df["internacoes_pediatricas_graves"].rank(pct=True)
        + df["dias_uti_pediatrica"].rank(pct=True)
        + df["populacao_0_14"].rank(pct=True)
        - df["uti_pediatrica_por_1000_criancas"].rank(pct=True)
    )
    .sort_values("sinal_uti_pediatrica_exploratorio", ascending=False)
    [
        [
            "municipio_nome",
            "regiao_saude",
            "populacao_0_14",
            "internacoes_pediatricas_graves",
            "internacoes_pediatricas_com_uti",
            "dias_uti_pediatrica",
            "mortalidade_pediatrica_hospitalar",
            "uti_pediatrica_por_1000_criancas",
            "sinal_uti_pediatrica_exploratorio",
        ]
    ]
    .head(20)
)

## Features específicas: UTI Neonatal

A tabela `features_uti_neonatal.parquet` concentra sinais próprios da análise de UTI Neonatal SUS.

A leitura combina nascidos vivos, prematuridade, baixo peso ao nascer, mortalidade neonatal, internações neonatais, dias de UTI em neonatos e oferta SUS de UTI Neonatal. Esses sinais são especialmente sensíveis à qualidade dos registros materno-infantis e ao fato de SINASC/SIM terem disponibilidade anual fechada até 2024 no workspace atual.

In [ ]:
features_uti_neonatal.shape

In [ ]:
features_uti_neonatal.head(10)

In [ ]:
uti_neonatal_catalog = pd.DataFrame(
    [
        {
            "feature": "nascidos_vivos",
            "grupo": "demanda/denominador",
            "interpretação": "Total anual de nascidos vivos por município de residência da mãe.",
            "fonte": "SINASC",
            "cuidado": "É o denominador central para indicadores neonatais; 2025 pode ficar nulo se o SINASC final ainda não estiver processado.",
        },
        {
            "feature": "prematuridade",
            "grupo": "risco neonatal",
            "interpretação": "Proporção de nascidos vivos prematuros entre os nascidos vivos do município.",
            "fonte": "SINASC",
            "cuidado": "Depende de preenchimento de idade gestacional; é fator de risco, não demanda hospitalar direta isolada.",
        },
        {
            "feature": "baixo_peso_ao_nascer",
            "grupo": "risco neonatal",
            "interpretação": "Proporção de nascidos vivos com baixo peso ao nascer entre os nascidos vivos do município.",
            "fonte": "SINASC",
            "cuidado": "Usa peso menor que 2.500g; deve ser lido junto com prematuridade e rede materno-infantil regional.",
        },
        {
            "feature": "mortalidade_neonatal",
            "grupo": "desfecho populacional",
            "interpretação": "Óbitos neonatais por 1.000 nascidos vivos.",
            "fonte": "SIM + SINASC",
            "cuidado": "Não mede qualidade assistencial isolada; municípios pequenos podem ter alta volatilidade anual.",
        },
        {
            "feature": "internacoes_neonatais",
            "grupo": "demanda hospitalar",
            "interpretação": "Total anual de internações de residentes com idade menor que 28 dias.",
            "fonte": "SIH",
            "cuidado": "É uma proxy por idade no SIH; não identifica diretamente o tipo físico do leito ocupado.",
        },
        {
            "feature": "dias_uti_neonatal",
            "grupo": "carga assistencial intensiva",
            "interpretação": "Soma anual de dias de UTI em internações de residentes com idade menor que 28 dias e uso de UTI.",
            "fonte": "SIH",
            "cuidado": "Soma `dias_uti_mes` e `dias_uti_intercorrencia`; depende da qualidade de preenchimento do SIH.",
        },
        {
            "feature": "uti_neonatal_por_1000_nascidos_vivos",
            "grupo": "oferta",
            "interpretação": "Média anual de leitos SUS de UTI Neonatal por 1.000 nascidos vivos.",
            "fonte": "CNES + SINASC",
            "cuidado": "Usa QT_SUS médio mensal e códigos CNES mapeados como UTI Neonatal; não mede ocupação nem disponibilidade regulada.",
        },
    ]
)

uti_neonatal_catalog

### Cobertura e nulos UTI Neonatal

A tabela de UTI Neonatal tem uma linha por `município + ano`. As features dependentes de SINASC/SIM podem ficar nulas em `2025`, porque essas bases complementares disponíveis no workspace cobrem anos fechados até `2024`. Internações neonatais pelo SIH podem existir em `2025`, mas indicadores por nascidos vivos precisam aguardar denominador final.

In [ ]:
uti_neonatal_coverage = {
    "linhas": len(features_uti_neonatal),
    "municipios": features_uti_neonatal["municipio_id"].nunique(),
    "anos": sorted(features_uti_neonatal["ano"].unique().tolist()),
    "chave_duplicada": features_uti_neonatal.duplicated(["municipio_id", "ano"]).any(),
}

uti_neonatal_coverage

In [ ]:
(
    features_uti_neonatal.isna()
    .sum()
    .rename("valores_ausentes")
    .to_frame()
    .assign(percentual=lambda df: df["valores_ausentes"] / len(features_uti_neonatal))
)

### Leitura exploratória UTI Neonatal

O ranking abaixo é exploratório. Ele combina maior volume de nascidos vivos, maiores sinais de risco neonatal, maior carga de internações/dias de UTI e menor oferta relativa de UTI Neonatal SUS. Ainda não é o score final de priorização.

In [ ]:
features_uti_neonatal_named = features_uti_neonatal.merge(
    dim[["municipio_id", "municipio_nome", "regiao_saude", "macroregiao_saude"]],
    on="municipio_id",
    how="left",
    validate="many_to_one",
)

features_uti_neonatal_named.head()

In [ ]:
features_uti_neonatal_named.groupby("ano").agg(
    municipios=("municipio_id", "nunique"),
    nascidos_vivos=("nascidos_vivos", "sum"),
    media_prematuridade=("prematuridade", "mean"),
    media_baixo_peso=("baixo_peso_ao_nascer", "mean"),
    media_mortalidade_neonatal=("mortalidade_neonatal", "mean"),
    internacoes_neonatais=("internacoes_neonatais", "sum"),
    dias_uti_neonatal=("dias_uti_neonatal", "sum"),
    media_uti_neonatal_1000_nascidos=("uti_neonatal_por_1000_nascidos_vivos", "mean"),
)

In [ ]:
ano_uti_neonatal = int(features_uti_neonatal_named["ano"].max())
if features_uti_neonatal_named.query("ano == @ano_uti_neonatal")["nascidos_vivos"].notna().sum() == 0:
    ano_uti_neonatal = int(features_uti_neonatal_named.loc[features_uti_neonatal_named["nascidos_vivos"].notna(), "ano"].max())

(
    features_uti_neonatal_named.query("ano == @ano_uti_neonatal")
    .assign(
        sinal_uti_neonatal_exploratorio=lambda df: df["nascidos_vivos"].rank(pct=True)
        + df["prematuridade"].rank(pct=True)
        + df["baixo_peso_ao_nascer"].rank(pct=True)
        + df["internacoes_neonatais"].rank(pct=True)
        + df["dias_uti_neonatal"].rank(pct=True)
        - df["uti_neonatal_por_1000_nascidos_vivos"].rank(pct=True)
    )
    .sort_values("sinal_uti_neonatal_exploratorio", ascending=False)
    [
        [
            "municipio_nome",
            "regiao_saude",
            "nascidos_vivos",
            "prematuridade",
            "baixo_peso_ao_nascer",
            "mortalidade_neonatal",
            "internacoes_neonatais",
            "dias_uti_neonatal",
            "uti_neonatal_por_1000_nascidos_vivos",
            "sinal_uti_neonatal_exploratorio",
        ]
    ]
    .head(20)
)

## Features regionais: UTI Queimado

A tabela `features_uti_queimado_regional.parquet` não usa unidade municipal. A unidade analítica é `regiao_saude + macroregiao_saude + ano`, porque UTI de queimados é rara, especializada e não deve ser lida como algo que todo município poderia ou deveria receber.

A leitura combina internações por queimaduras, óbitos por queimaduras, distância regional até uma referência SUS com UTI de queimados, população regional e histórico de transferências. Esses sinais servem para investigação regional e planejamento de rede, não para prescrição automática de leitos.

In [ ]:
features_uti_queimado_regional.shape

In [ ]:
features_uti_queimado_regional.head(10)

In [ ]:
uti_queimado_catalog = pd.DataFrame(
    [
        {
            "feature": "internacoes_por_queimaduras",
            "grupo": "demanda regional",
            "interpretação": "Total anual de internações por queimaduras de residentes da região de saúde.",
            "fonte": "SIH",
            "cuidado": "Usa diagnóstico CID-10 T20-T32 ou X00-X19 como proxy de queimadura; deve ser lido em escala regional.",
        },
        {
            "feature": "obitos_por_queimaduras",
            "grupo": "desfecho regional",
            "interpretação": "Total anual de óbitos por queimaduras de residentes da região de saúde.",
            "fonte": "SIM",
            "cuidado": "Fica nulo quando o ano do SIM ainda não está disponível; não preencher como zero nesses casos.",
        },
        {
            "feature": "distancia_ate_referencia_queimados",
            "grupo": "acesso especializado",
            "interpretação": "Distância em km do centroide simples da região até o município mais próximo com UTI de queimados SUS no ano.",
            "fonte": "CNES + dimensão municipal",
            "cuidado": "É distância geográfica em linha reta; não substitui tempo de deslocamento, regulação ou desenho formal da rede.",
        },
        {
            "feature": "populacao_regional",
            "grupo": "denominador regional",
            "interpretação": "Soma da população dos municípios da região de saúde no ano.",
            "fonte": "IBGE + dimensão municipal",
            "cuidado": "Ajuda a dimensionar exposição regional, mas queimaduras graves são eventos raros e concentrados.",
        },
        {
            "feature": "historico_transferencias",
            "grupo": "fluxo regional",
            "interpretação": "Internações por queimaduras em que município de residência e município de internação são diferentes.",
            "fonte": "SIH",
            "cuidado": "É proxy de deslocamento/transferência; não identifica formalmente regulação, vaga negada ou transporte sanitário.",
        },
    ]
)

uti_queimado_catalog

### Cobertura e nulos UTI Queimado

A tabela regional cobre regiões de saúde por ano. Como SIM disponível no workspace cobre anos fechados até `2024`, `obitos_por_queimaduras` pode ficar nulo em `2025`. Esse nulo é metodológico: significa ausência da base final, não ausência de óbitos.

In [ ]:
uti_queimado_coverage = {
    "linhas": len(features_uti_queimado_regional),
    "regioes_saude": features_uti_queimado_regional["regiao_saude"].nunique(),
    "macroregioes_saude": features_uti_queimado_regional["macroregiao_saude"].nunique(),
    "anos": sorted(features_uti_queimado_regional["ano"].unique().tolist()),
    "chave_duplicada": features_uti_queimado_regional.duplicated(["regiao_saude", "macroregiao_saude", "ano"]).any(),
}

uti_queimado_coverage

In [ ]:
(
    features_uti_queimado_regional.isna()
    .sum()
    .rename("valores_ausentes")
    .to_frame()
    .assign(percentual=lambda df: df["valores_ausentes"] / len(features_uti_queimado_regional))
)

### Leitura exploratória UTI Queimado

O ranking abaixo é exploratório e regional. Ele combina maior volume de internações por queimaduras, óbitos por queimaduras, transferências, população regional e maior distância até referência especializada. Ainda não é score final nem indicação automática de implantação de UTI de queimados.

In [ ]:
features_uti_queimado_regional.groupby("ano").agg(
    regioes=("regiao_saude", "nunique"),
    internacoes_por_queimaduras=("internacoes_por_queimaduras", "sum"),
    obitos_por_queimaduras=("obitos_por_queimaduras", "sum"),
    historico_transferencias=("historico_transferencias", "sum"),
    populacao_regional=("populacao_regional", "sum"),
    distancia_media_referencia=("distancia_ate_referencia_queimados", "mean"),
)

In [ ]:
ano_uti_queimado = int(features_uti_queimado_regional["ano"].max())
if features_uti_queimado_regional.query("ano == @ano_uti_queimado")["obitos_por_queimaduras"].notna().sum() == 0:
    ano_uti_queimado = int(features_uti_queimado_regional.loc[features_uti_queimado_regional["obitos_por_queimaduras"].notna(), "ano"].max())

(
    features_uti_queimado_regional.query("ano == @ano_uti_queimado")
    .assign(
        sinal_uti_queimado_exploratorio=lambda df: df["internacoes_por_queimaduras"].rank(pct=True)
        + df["obitos_por_queimaduras"].rank(pct=True)
        + df["historico_transferencias"].rank(pct=True)
        + df["populacao_regional"].rank(pct=True)
        + df["distancia_ate_referencia_queimados"].rank(pct=True)
    )
    .sort_values("sinal_uti_queimado_exploratorio", ascending=False)
    [
        [
            "regiao_saude",
            "macroregiao_saude",
            "internacoes_por_queimaduras",
            "obitos_por_queimaduras",
            "distancia_ate_referencia_queimados",
            "populacao_regional",
            "historico_transferencias",
            "sinal_uti_queimado_exploratorio",
        ]
    ]
    .head(20)
)

## Features específicas: UTI Coronariana

A tabela `features_uti_coronariana.parquet` concentra sinais próprios da análise de UTI Coronariana SUS.

A leitura combina internações por infarto, internações cardiovasculares, óbitos cardiovasculares, população idosa, dias de UTI em internações cardiovasculares, oferta SUS de leitos coronarianos e distância até referência de cardiologia. Esses sinais apoiam investigação da rede cardiovascular, não uma indicação automática de implantação de leitos.

In [ ]:
features_uti_coronariana.shape

In [ ]:
features_uti_coronariana.head(10)

In [ ]:
uti_coronariana_catalog = pd.DataFrame(
    [
        {
            "feature": "internacoes_por_infarto",
            "grupo": "demanda cardiovascular específica",
            "interpretação": "Total anual de internações de residentes com diagnóstico principal CID-10 I21 ou I22.",
            "fonte": "SIH",
            "cuidado": "É proxy de infarto agudo/subsequente no SIH; não substitui linha de cuidado cardiológica formal.",
        },
        {
            "feature": "internacoes_cardiovasculares",
            "grupo": "demanda cardiovascular ampla",
            "interpretação": "Total anual de internações de residentes com diagnóstico principal CID-10 iniciado por I.",
            "fonte": "SIH",
            "cuidado": "Sinal amplo de carga cardiovascular; nem toda internação cardiovascular demanda UTI coronariana.",
        },
        {
            "feature": "obitos_cardiovasculares",
            "grupo": "desfecho populacional",
            "interpretação": "Total anual de óbitos cardiovasculares por município de residência.",
            "fonte": "SIM",
            "cuidado": "Fica nulo quando o ano do SIM ainda não está disponível; não preencher como zero nesses casos.",
        },
        {
            "feature": "populacao_60_mais",
            "grupo": "vulnerabilidade/denominador",
            "interpretação": "População estimada de 60 anos ou mais no município.",
            "fonte": "IBGE",
            "cuidado": "A idade é um fator de risco importante, mas não deve dominar a leitura sem demanda e acesso regional.",
        },
        {
            "feature": "dias_uti_coronariana",
            "grupo": "carga assistencial intensiva",
            "interpretação": "Soma anual de dias de UTI em internações cardiovasculares com uso de UTI.",
            "fonte": "SIH",
            "cuidado": "SIH registra uso de UTI, mas não identifica diretamente o tipo físico do leito ocupado.",
        },
        {
            "feature": "leitos_coronarianos_por_1000_idosos",
            "grupo": "oferta",
            "interpretação": "Média anual de leitos SUS de UTI Coronariana por 1.000 pessoas de 60 anos ou mais.",
            "fonte": "CNES + IBGE",
            "cuidado": "Usa QT_SUS médio mensal e códigos CNES mapeados como UCO/UTI Coronariana; não mede ocupação nem disponibilidade regulada.",
        },
        {
            "feature": "distancia_ate_referencia_cardiologia",
            "grupo": "acesso especializado",
            "interpretação": "Distância em km até o município mais próximo com leito SUS de UTI Coronariana no ano.",
            "fonte": "CNES + dimensão municipal",
            "cuidado": "É distância geográfica em linha reta; não substitui tempo de deslocamento, SAMU, regulação ou rede pactuada.",
        },
    ]
)

uti_coronariana_catalog

### Cobertura e nulos UTI Coronariana

A tabela de UTI Coronariana tem uma linha por `município + ano`. Como SIM disponível no workspace cobre anos fechados até `2024`, `obitos_cardiovasculares` pode ficar nulo em `2025`. Esse nulo é metodológico: significa ausência da base final, não ausência de óbitos.

In [ ]:
uti_coronariana_coverage = {
    "linhas": len(features_uti_coronariana),
    "municipios": features_uti_coronariana["municipio_id"].nunique(),
    "anos": sorted(features_uti_coronariana["ano"].unique().tolist()),
    "chave_duplicada": features_uti_coronariana.duplicated(["municipio_id", "ano"]).any(),
}

uti_coronariana_coverage

In [ ]:
(
    features_uti_coronariana.isna()
    .sum()
    .rename("valores_ausentes")
    .to_frame()
    .assign(percentual=lambda df: df["valores_ausentes"] / len(features_uti_coronariana))
)

### Leitura exploratória UTI Coronariana

O ranking abaixo é exploratório. Ele combina maior carga cardiovascular, maior população idosa, maior distância até referência cardiológica e menor oferta relativa de leitos coronarianos SUS. Ainda não é o score final de priorização.

In [ ]:
features_uti_coronariana_named = features_uti_coronariana.merge(
    dim[["municipio_id", "municipio_nome", "regiao_saude", "macroregiao_saude"]],
    on="municipio_id",
    how="left",
    validate="many_to_one",
)

features_uti_coronariana_named.head()

In [ ]:
features_uti_coronariana_named.groupby("ano").agg(
    municipios=("municipio_id", "nunique"),
    internacoes_por_infarto=("internacoes_por_infarto", "sum"),
    internacoes_cardiovasculares=("internacoes_cardiovasculares", "sum"),
    obitos_cardiovasculares=("obitos_cardiovasculares", "sum"),
    populacao_60_mais=("populacao_60_mais", "sum"),
    dias_uti_coronariana=("dias_uti_coronariana", "sum"),
    media_leitos_coronarianos_1000_idosos=("leitos_coronarianos_por_1000_idosos", "mean"),
    distancia_media_cardiologia=("distancia_ate_referencia_cardiologia", "mean"),
)

In [ ]:
ano_uti_coronariana = int(features_uti_coronariana_named["ano"].max())
if features_uti_coronariana_named.query("ano == @ano_uti_coronariana")["obitos_cardiovasculares"].notna().sum() == 0:
    ano_uti_coronariana = int(features_uti_coronariana_named.loc[features_uti_coronariana_named["obitos_cardiovasculares"].notna(), "ano"].max())

(
    features_uti_coronariana_named.query("ano == @ano_uti_coronariana")
    .assign(
        sinal_uti_coronariana_exploratorio=lambda df: df["internacoes_por_infarto"].rank(pct=True)
        + df["internacoes_cardiovasculares"].rank(pct=True)
        + df["obitos_cardiovasculares"].rank(pct=True)
        + df["populacao_60_mais"].rank(pct=True)
        + df["dias_uti_coronariana"].rank(pct=True)
        + df["distancia_ate_referencia_cardiologia"].rank(pct=True)
        - df["leitos_coronarianos_por_1000_idosos"].rank(pct=True)
    )
    .sort_values("sinal_uti_coronariana_exploratorio", ascending=False)
    [
        [
            "municipio_nome",
            "regiao_saude",
            "internacoes_por_infarto",
            "internacoes_cardiovasculares",
            "obitos_cardiovasculares",
            "populacao_60_mais",
            "dias_uti_coronariana",
            "leitos_coronarianos_por_1000_idosos",
            "distancia_ate_referencia_cardiologia",
            "sinal_uti_coronariana_exploratorio",
        ]
    ]
    .head(20)
)

## Checks rápidos de sanidade

Esses checks não provam que a metodologia está perfeita, mas ajudam a detectar erros grosseiros de ETL ou feature engineering.

In [19]:
partos_preenchidos = features_obstetricos["partos_12m"].notna()

checks = {
    "comum_sem_chave_duplicada": not features.duplicated(["municipio_id", "ano", "tipo_leito"]).any(),
    "comum_populacao_positiva": features["populacao_total"].gt(0).all(),
    "comum_leitos_nao_negativos": features["leitos_sus_por_1000_hab"].ge(0).all(),
    "comum_internacoes_nao_negativas": features["internacoes_por_1000_hab"].ge(0).all(),
    "comum_dias_nao_negativos": features["dias_permanencia_por_1000_hab"].ge(0).all(),
    "comum_distancia_nao_negativa": features["distancia_ate_polo_hospitalar"].ge(0).all(),
    "comum_mortalidade_entre_0_e_1_quando_preenchida": features["taxa_mortalidade_hospitalar"].dropna().between(0, 1).all(),
    "clinico_sem_chave_duplicada": not features_clinicos.duplicated(["municipio_id", "ano"]).any(),
    "clinico_internacoes_nao_negativas": features_clinicos["internacoes_clinicas_12m"].ge(0).all(),
    "clinico_dias_nao_negativos": features_clinicos["dias_clinicos_permanencia"].ge(0).all(),
    "clinico_percentual_idosos_entre_0_e_1": features_clinicos["percentual_idosos"].dropna().between(0, 1).all(),
    "clinico_leitos_nao_negativos": features_clinicos["leitos_clinicos_por_1000_hab"].ge(0).all(),
    "cirurgico_sem_chave_duplicada": not features_cirurgicos.duplicated(["municipio_id", "ano"]).any(),
    "cirurgico_internacoes_nao_negativas": features_cirurgicos["internacoes_cirurgicas_12m"].ge(0).all(),
    "cirurgico_procedimentos_nao_negativos": features_cirurgicos["procedimentos_cirurgicos_12m"].ge(0).all(),
    "cirurgico_internacoes_iguais_a_procedimentos_proxy": (features_cirurgicos["internacoes_cirurgicas_12m"] == features_cirurgicos["procedimentos_cirurgicos_12m"]).all(),
    "cirurgico_permanencia_nao_negativa": features_cirurgicos["tempo_medio_permanencia_cirurgico"].dropna().ge(0).all(),
    "cirurgico_valor_medio_nao_negativo": features_cirurgicos["valor_medio_cirurgico"].dropna().ge(0).all(),
    "cirurgico_leitos_nao_negativos": features_cirurgicos["leitos_cirurgicos_por_1000_hab"].ge(0).all(),
    "cirurgico_centros_nao_negativos": features_cirurgicos["estabelecimentos_com_centro_cirurgico"].ge(0).all(),
    "obstetrico_sem_chave_duplicada": not features_obstetricos.duplicated(["municipio_id", "ano"]).any(),
    "obstetrico_mulheres_10_49_positiva": features_obstetricos["mulheres_10_49"].gt(0).all(),
    "obstetrico_nascidos_nao_negativos_quando_preenchido": features_obstetricos["nascidos_vivos_12m"].dropna().ge(0).all(),
    "obstetrico_partos_nao_negativos_quando_preenchido": features_obstetricos["partos_12m"].dropna().ge(0).all(),
    "obstetrico_partos_somam_cesareos_vaginais": (features_obstetricos.loc[partos_preenchidos, "partos_12m"] == (features_obstetricos.loc[partos_preenchidos, "partos_cesareos"] + features_obstetricos.loc[partos_preenchidos, "partos_vaginais"])).all(),
    "obstetrico_leitos_nao_negativos": features_obstetricos["leitos_obstetricos_por_1000_mulheres_10_49"].ge(0).all(),
    "obstetrico_distancia_nao_negativa": features_obstetricos["distancia_ate_maternidade_referencia"].ge(0).all(),
    "uti_adulto_sem_chave_duplicada": not features_uti_adulto.duplicated(["municipio_id", "ano"]).any(),
    "uti_adulto_internacoes_nao_negativas": features_uti_adulto["internacoes_com_uti_adulto"].ge(0).all(),
    "uti_adulto_populacao_60_mais_nao_negativa": features_uti_adulto["populacao_60_mais"].dropna().ge(0).all(),
    "uti_adulto_mortalidade_entre_0_e_1_quando_preenchida": features_uti_adulto["mortalidade_hospitalar_adulta"].dropna().between(0, 1).all(),
    "uti_adulto_cardiovasculares_nao_negativas": features_uti_adulto["internacoes_cardiovasculares"].ge(0).all(),
    "uti_adulto_respiratorias_nao_negativas": features_uti_adulto["internacoes_respiratorias"].ge(0).all(),
    "uti_adulto_dias_nao_negativos": features_uti_adulto["dias_uti_adulto"].ge(0).all(),
    "uti_adulto_leitos_nao_negativos": features_uti_adulto["uti_adulto_por_1000_adultos"].ge(0).all(),
    "uti_pediatrica_sem_chave_duplicada": not features_uti_pediatrica.duplicated(["municipio_id", "ano"]).any(),
    "uti_pediatrica_populacao_0_14_nao_negativa": features_uti_pediatrica["populacao_0_14"].dropna().ge(0).all(),
    "uti_pediatrica_graves_nao_negativas": features_uti_pediatrica["internacoes_pediatricas_graves"].ge(0).all(),
    "uti_pediatrica_com_uti_nao_negativas": features_uti_pediatrica["internacoes_pediatricas_com_uti"].ge(0).all(),
    "uti_pediatrica_graves_maior_igual_com_uti": (features_uti_pediatrica["internacoes_pediatricas_graves"] >= features_uti_pediatrica["internacoes_pediatricas_com_uti"]).all(),
    "uti_pediatrica_dias_nao_negativos": features_uti_pediatrica["dias_uti_pediatrica"].ge(0).all(),
    "uti_pediatrica_mortalidade_entre_0_e_1_quando_preenchida": features_uti_pediatrica["mortalidade_pediatrica_hospitalar"].dropna().between(0, 1).all(),
    "uti_pediatrica_leitos_nao_negativos": features_uti_pediatrica["uti_pediatrica_por_1000_criancas"].ge(0).all(),
    "uti_neonatal_sem_chave_duplicada": not features_uti_neonatal.duplicated(["municipio_id", "ano"]).any(),
    "uti_neonatal_nascidos_nao_negativos_quando_preenchido": features_uti_neonatal["nascidos_vivos"].dropna().ge(0).all(),
    "uti_neonatal_prematuridade_entre_0_e_1": features_uti_neonatal["prematuridade"].dropna().between(0, 1).all(),
    "uti_neonatal_baixo_peso_entre_0_e_1": features_uti_neonatal["baixo_peso_ao_nascer"].dropna().between(0, 1).all(),
    "uti_neonatal_mortalidade_nao_negativa": features_uti_neonatal["mortalidade_neonatal"].dropna().ge(0).all(),
    "uti_neonatal_internacoes_nao_negativas": features_uti_neonatal["internacoes_neonatais"].ge(0).all(),
    "uti_neonatal_dias_nao_negativos": features_uti_neonatal["dias_uti_neonatal"].ge(0).all(),
    "uti_neonatal_leitos_nao_negativos": features_uti_neonatal["uti_neonatal_por_1000_nascidos_vivos"].dropna().ge(0).all(),
    "uti_queimado_regional_sem_chave_duplicada": not features_uti_queimado_regional.duplicated(["regiao_saude", "macroregiao_saude", "ano"]).any(),
    "uti_queimado_internacoes_nao_negativas": features_uti_queimado_regional["internacoes_por_queimaduras"].ge(0).all(),
    "uti_queimado_obitos_nao_negativos_quando_preenchido": features_uti_queimado_regional["obitos_por_queimaduras"].dropna().ge(0).all(),
    "uti_queimado_distancia_nao_negativa": features_uti_queimado_regional["distancia_ate_referencia_queimados"].dropna().ge(0).all(),
    "uti_queimado_populacao_regional_positiva": features_uti_queimado_regional["populacao_regional"].gt(0).all(),
    "uti_queimado_transferencias_nao_negativas": features_uti_queimado_regional["historico_transferencias"].ge(0).all(),
    "uti_queimado_transferencias_ate_internacoes": (features_uti_queimado_regional["historico_transferencias"] <= features_uti_queimado_regional["internacoes_por_queimaduras"]).all(),
    "uti_coronariana_sem_chave_duplicada": not features_uti_coronariana.duplicated(["municipio_id", "ano"]).any(),
    "uti_coronariana_infarto_nao_negativas": features_uti_coronariana["internacoes_por_infarto"].ge(0).all(),
    "uti_coronariana_cardio_nao_negativas": features_uti_coronariana["internacoes_cardiovasculares"].ge(0).all(),
    "uti_coronariana_infarto_ate_cardio": (features_uti_coronariana["internacoes_por_infarto"] <= features_uti_coronariana["internacoes_cardiovasculares"]).all(),
    "uti_coronariana_obitos_nao_negativos_quando_preenchido": features_uti_coronariana["obitos_cardiovasculares"].dropna().ge(0).all(),
    "uti_coronariana_populacao_60_mais_nao_negativa": features_uti_coronariana["populacao_60_mais"].dropna().ge(0).all(),
    "uti_coronariana_dias_nao_negativos": features_uti_coronariana["dias_uti_coronariana"].ge(0).all(),
    "uti_coronariana_leitos_nao_negativos": features_uti_coronariana["leitos_coronarianos_por_1000_idosos"].ge(0).all(),
    "uti_coronariana_distancia_nao_negativa": features_uti_coronariana["distancia_ate_referencia_cardiologia"].dropna().ge(0).all(),
}

pd.Series(checks, name="ok")

{'sem_chave_duplicada': True,
 'populacao_positiva': True,
 'leitos_nao_negativos': True,
 'internacoes_nao_negativas': True,
 'dias_nao_negativos': True,
 'distancia_nao_negativa': True,
 'mortalidade_entre_0_e_1_quando_preenchida': True}

## Leitura metodológica

A camada comum, as camadas específicas municipais e a primeira camada regional para UTI Queimado estão prontas para exploração e para alimentar um baseline de score transparente.

Pontos fortes:

- unidade analítica clara;
- features simples de auditar;
- separação por tipo de leito;
- preservação de fluxo residência versus internação;
- inclusão de sinais específicos para UTI Adulto, como uso de UTI, dias de UTI, mortalidade hospitalar adulta e causas cardiovasculares/respiratórias;
- inclusão de sinais específicos para UTI Pediátrica, como população infantil, internações graves, uso de UTI, dias de UTI e mortalidade hospitalar pediátrica;
- inclusão de sinais específicos para UTI Neonatal, como nascidos vivos, prematuridade, baixo peso, mortalidade neonatal e oferta por nascidos vivos;
- tratamento regional da UTI de queimados, respeitando raridade e alta especialização do serviço;
- inclusão de sinais específicos para UTI Coronariana, como infarto, carga cardiovascular, população idosa, dias de UTI e distância até referência cardiológica;
- linguagem compatível com investigação, não prescrição automática.

Limitações atuais:

- SIH por tipo de leito ainda é proxy;
- distância é geográfica, não tempo real de deslocamento;
- população total é denominador comum na camada geral; nas camadas específicas já entram recortes como população 60+, centro cirúrgico, mulheres de 10 a 49 anos, adulto proxy para UTI Adulto, população de 0 a 14 anos para UTI Pediátrica, nascidos vivos para UTI Neonatal e população regional para UTI Queimado;
- SINASC/SIM de 2025 podem permanecer nulos até haver ano final processado;
- histórico de transferências é proxy por município de residência diferente do município de internação, não registro formal de regulação;
- ausência de leito local não deve virar regra automática de prioridade;
- as features ainda não incorporam pesos, normalização final ou score.

Próximo passo natural: iniciar o baseline transparente de score por tipo de leito, separando explicitamente scores municipais e regionais.